# VSL 30 từ — V4.3 FINAL Run-All (train → evaluate → export)

V4.3 tiếp tục trực tiếp từ V4.2 và giữ các phần đã có tín hiệu tốt:

- **Main head only** cho prediction cuối; không fusion metric/hand head.
- **Không Stage 3**; giữ Stage 1 warm-up + Stage 2 fine-tune.
- **Giữ clean + deterministic stress validation** để checkpoint bớt overfit random validation.
- **Giảm pair-specific pressure `Thích ↔ Muốn`** để tránh đẩy lỗi sang class khác.
- Thay phần hard-pair margin diện rộng bằng **hard-negative metric learning trên embedding** cho cụm khó `Thích/Muốn/Ăn/Bố/Anh`.
- **Sampling boost nhẹ, không oversample mạnh**.
- Checkpoint selection không chỉ nhìn overall Macro-F1 mà còn nhìn **mean/min F1 của hard cluster** để tránh một class khó bị collapse.
- Không train/adapt 472 class. Không RGB. Không horizontal flip. Test không dùng để tune threshold/fusion trong chính run này.

> Lưu ý phương pháp: V4.3 được thiết kế sau khi đã xem kết quả test của V4.2. Vì vậy bộ test hiện tại nên được xem là **development test** khi so các phiên bản. Muốn báo cáo accuracy cuối cùng khách quan, nên dùng thêm một holdout/unseen-signer chưa từng dùng để quyết định kiến trúc/loss.


> **Kaggle runtime fix:** notebook tự kiểm tra `pandas`, `scikit-learn`, `matplotlib`, `tqdm`; package nào thiếu mới được cài trước khi import.


## Run-All behavior

Notebook này được thiết kế để chạy **một mạch từ đầu đến cuối**:

1. Load dataset VSL v2 + pretrained encoder `best_vsl_metric_encoder.pt`.
2. Build đúng 30 glosses và train lại V4.3.
3. Chọn best checkpoint bằng clean/stress/hard-class validation.
4. Load best checkpoint của **chính lần chạy hiện tại**.
5. Test/evaluate.
6. Export tự động:
   - `best_vsl30_v4_3.pt`
   - `vsl30_v4_3_main_head_state.pt`
   - `vsl30_v4_3_main_head.ts` (TorchScript)
   - `vsl30_v4_3_main.onnx` (ONNX)
   - `label_map.json`
   - `deployment_config.json`
   - `vsl30_v4_3_FINAL_bundle.zip`

**Không cần add checkpoint V4.3 cũ.** Vẫn cần pretrained encoder cũ `best_vsl_metric_encoder.pt`.


In [ ]:
# ============================================================
# 0. CONFIG — V4.3 30-CLASS ONLY
# ============================================================
from pathlib import Path

SEED = 42

TARGET_GLOSSES = [
    "Em", "Anh", "Chị", "Bố", "Nghe", "Nói", "Đi", "Ăn", "Thích", "Muốn",
    "Cần", "Cho", "Giúp đỡ", "Làm việc", "Nghỉ ngơi", "Thức dậy", "Tiếp tục",
    "Đồng ý", "Từ chối", "Cảm ơn", "Xin lỗi", "Nhà", "Trường học", "Bệnh viện",
    "Thành phố", "Siêu thị", "Bây giờ", "Ngày", "Sớm", "Nên",
]

# User-supplied blacklist: 87 entries, 86 unique IDs.
BAD_VIDEO_IDS = {
    "Xin lỗi": {485896, 455505},
    "Nghỉ ngơi": {343467, 265649, 348728, 702733, 810526, 994787},
    "Thức dậy": {139581, 244247, 324597, 410160, 774751, 794083, 836971, 903373, 903531, 985447},
    "Cảm ơn": {266931, 526117, 687080, 835343, 975136, 255287},
    "Nghe": {158845, 203225, 252640, 278240},
    "Nói": {204180},
    "Đi": {276827, 644103, 772612, 843778, 855638, 983456},
    "Nhà": {940886, 665610, 736553, 310548, 243423},
    "Trường học": {745239, 738457, 624108, 415779, 451577, 350021, 245534, 140163},
    "Ăn": {183667, 271339, 319684, 330283, 379946, 660086, 738720, 765822},
    "Bệnh viện": {156802, 446479, 544154, 858490},
    "Đồng ý": {
        162823, 176517, 250647, 265759, 313933, 321048, 356773, 403742,
        416768, 427766, 525818, 535810, 536822, 560959, 587002, 606404,
        622230, 672079, 748133, 770030, 773088, 818264, 822063, 853258,
        904214, 946567,
    },
}

DATASET_HINT = "vsl-vietnamese-sign-language-v2"
PRETRAIN_NAME = "best_vsl_metric_encoder.pt"
REQUIRE_PRETRAIN = True

# Input
SEQ_LEN = 48
KP_USED_POINTS = 75
N_POSE = 33
N_HAND = 21
COORD_CLIP = 8.0
MIN_SHOULDER_SCALE = 0.08
MIN_HAND_SCALE = 0.04
LOCAL_HAND_CLIP = 6.0

# Split / data safety
VAL_FRACTION = 0.15
TRY_SIGNER_AWARE_VAL = True
SIGNER_MAP_MIN_COVERAGE = 0.60
MIN_AFTER_FILTER_WARNING = 60
USE_AUGMENTED_KEYPOINTS = True
MAX_AUG_PER_ORIGINAL = 2

# Preprocessing
USE_KP_RECONSTRUCTION = True
RECON_MIN_OBS = 2
USE_ACTIVE_TRIM = False
ACTIVE_TRIM_MAX_FRAC = 0.08

# On-the-fly train augmentation. No horizontal flip.
TEMP_SPEED_MIN = 0.82
TEMP_SPEED_MAX = 1.18
TEMP_CROP_MIN_KEEP = 0.90
ROT_DEG = 6.0
SCALE_MIN = 0.94
SCALE_MAX = 1.06
TRANS_STD = 0.012
KP_NOISE_STD = 0.0035
POINT_DROP = 0.002
FRAME_DROP = 0.025
ALREADY_AUG_EXTRA_PROB = 0.35

# Deterministic stress validation: milder than train augmentation.
# Used only for checkpoint selection; never mixed into training.
USE_STRESS_VAL = True
STRESS_VAL_WEIGHT = 0.20
STRESS_SPEED_MIN = 0.92
STRESS_SPEED_MAX = 1.08
STRESS_CROP_MIN_KEEP = 0.95
STRESS_ROT_DEG = 4.0
STRESS_SCALE_MIN = 0.97
STRESS_SCALE_MAX = 1.03
STRESS_TRANS_STD = 0.006
STRESS_NOISE_STD = 0.0020
STRESS_POINT_DROP = 0.001
STRESS_FRAME_DROP = 0.010

# Model — old tensor shapes remain compatible with pretrained encoder.
REGION_DIM = 96
D_MODEL = 256
NHEAD = 8
NUM_TRANSFORMER_LAYERS = 3
FF_DIM = 512
DROPOUT = 0.12
EMBED_DIM = 256
HAND_TEMP_DIM = 128
USE_HAND_ANGLE_FEATURES = True
USE_GEOMETRY_RESIDUAL = True

# P×K batches
PK_P = 10
PK_K = 4
BATCHES_PER_EPOCH = 130

# Fine-tuning: V4.3 still stops after Stage 2.
STAGE1_EPOCHS = 4
STAGE2_EPOCHS = 10
EARLY_STOP_PATIENCE = 4
LR_NEW_STAGE1 = 3e-4
LR_OLD_STAGE2 = 2.5e-5
LR_NEW_STAGE2 = 1.3e-4
WEIGHT_DECAY = 1e-4

# Base auxiliary losses
LABEL_SMOOTHING = 0.03
SUPCON_WEIGHT = 0.15
KEYPOINT_AUX_WEIGHT = 0.08
HAND_AUX_WEIGHT = 0.08
HAND_SUPCON_WEIGHT = 0.04
METRIC_CE_WEIGHT = 0.10
SUPCON_TEMP = 0.08
METRIC_SCALE = 12.0
METRIC_MARGIN = 0.06

# V4.3: retain a small Thích-vs-Muốn boundary term, but no longer let it dominate.
FOCUS_PAIR = ("Thích", "Muốn")
FOCUS_PAIR_CE_WEIGHT = 0.10
FOCUS_PAIR_MARGIN_WEIGHT = 0.01
FOCUS_PAIR_MARGIN = 0.10

# Development hard cluster from prior V4.x runs.
# Hard-negative metric loss operates on normalized embeddings, not final output fusion.
HARD_CLUSTER = ["Thích", "Muốn", "Ăn", "Bố", "Anh"]
HARD_NEGATIVE_MAP = {
    "Thích": ["Muốn", "Anh", "Bố"],
    "Muốn": ["Thích", "Anh", "Bố"],
    "Ăn": ["Bố", "Anh"],
    "Bố": ["Ăn", "Anh", "Muốn"],
    "Anh": ["Bố", "Muốn", "Thích"],
}
HARD_NEG_METRIC_WEIGHT = 0.08
HARD_NEG_MARGIN = 0.12
HARD_NEG_TOPK = 2

# Mild sampling only. Values are class-selection weights in P×K sampling.
HARD_CLASS_SAMPLING_BOOSTS = {
    "Thích": 1.30,
    "Muốn": 1.30,
    "Ăn": 1.20,
    "Bố": 1.20,
    "Anh": 1.15,
}

# Checkpoint selection: overall quality remains dominant, but hard-class collapse is penalized.
SELECT_CLEAN_MACRO_W = 0.70
SELECT_STRESS_MACRO_W = 0.10
SELECT_HARD_MEAN_W = 0.15
SELECT_HARD_MIN_W = 0.05
assert abs(
    SELECT_CLEAN_MACRO_W + SELECT_STRESS_MACRO_W
    + SELECT_HARD_MEAN_W + SELECT_HARD_MIN_W - 1.0
) < 1e-9

# V3.1 reference — reporting only, never used to tune on test.
CURRENT_V3_1_ACC = 0.8833693304535637
CURRENT_V3_1_MACRO_F1 = 0.8835788969139767
CURRENT_V3_1_TOP3 = 0.9546436285097192

# Prior user-run references — reporting only.
CURRENT_V4_1_MAIN_ACC = 0.8833693304535637
CURRENT_V4_1_MAIN_MACRO_F1 = 0.8875067351415435
CURRENT_V4_1_MAIN_TOP3 = 0.9654427645788337
CURRENT_V4_2_ACC = 0.8833693304535637
CURRENT_V4_2_MACRO_F1 = 0.8861650580412004
CURRENT_V4_2_TOP3 = 0.9546436285097192

NUM_WORKERS = 0
MAX_GRAD_NORM = 1.0
CHECKPOINT_EPS = 1e-6

QUICK_MODE = False
if QUICK_MODE:
    BATCHES_PER_EPOCH = 10
    STAGE1_EPOCHS = STAGE2_EPOCHS = 1

WORK_DIR = Path("/kaggle/working/vsl30_v4_3")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("30 glosses:", len(TARGET_GLOSSES))
print("Focus pair:", FOCUS_PAIR)
print("Hard cluster:", HARD_CLUSTER)
print("Unique bad IDs:", sum(len(x) for x in BAD_VIDEO_IDS.values()))
print("WORK_DIR:", WORK_DIR)


In [ ]:
# ============================================================
# 1. ENVIRONMENT — dependency preflight + P100-safe
# ============================================================
import os, sys, subprocess, importlib
from importlib.util import find_spec
from importlib.metadata import version as pkg_version

# Kaggle images can occasionally be missing a normally preinstalled package.
# Install ONLY what is missing; do not reinstall packages that already work.
REQUIRED_RUNTIME_PACKAGES = {
    "pandas": "pandas==2.2.3",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    # Needed at the end of this same Run-All for ONNX export + verification.
    "onnx": "onnx",
    "onnxruntime": "onnxruntime",
    "onnxscript": "onnxscript",
}

missing_pip_specs = [
    pip_spec
    for import_name, pip_spec in REQUIRED_RUNTIME_PACKAGES.items()
    if find_spec(import_name) is None
]

if missing_pip_specs:
    print("Missing runtime packages:", missing_pip_specs)
    print("Installing only missing packages ...")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "--disable-pip-version-check", "--no-input", "-q",
            *missing_pip_specs,
        ])
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            "Không cài được dependency còn thiếu. "
            "Trong Kaggle hãy bật Settings > Internet = On rồi Run All lại. "
            f"Missing: {missing_pip_specs}"
        ) from e
    importlib.invalidate_caches()

still_missing = [
    name for name in REQUIRED_RUNTIME_PACKAGES
    if find_spec(name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Vẫn thiếu package sau bước bootstrap: " + ", ".join(still_missing)
    )

def gpu_name_before_torch():
    try:
        return subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True
        ).strip().splitlines()[0]
    except Exception:
        return ""

GPU_PRE = gpu_name_before_torch()
print("GPU before torch import:", GPU_PRE or "unknown")

if "P100" in GPU_PRE.upper():
    try:
        torch_pkg = pkg_version("torch")
    except Exception:
        torch_pkg = None

    print("Torch package before:", torch_pkg)

    if torch_pkg != "2.10.0+cu126":
        print("P100 detected -> installing torch 2.10.0 cu126 ...")
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall",
                "torch==2.10.0",
                "--index-url", "https://download.pytorch.org/whl/cu126"
            ])
        except subprocess.CalledProcessError as e:
            raise RuntimeError(
                "Không cài được PyTorch build cho P100. "
                "Bật Kaggle Internet rồi chạy lại."
            ) from e

import json, math, random, re, unicodedata, warnings
from collections import defaultdict, Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    confusion_matrix, classification_report, top_k_accuracy_score
)

warnings.filterwarnings("ignore")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", sys.version.split()[0])
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("DEVICE:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("Compiled archs:", torch.cuda.get_arch_list())

    cap = torch.cuda.get_device_capability(0)
    arch = f"sm_{cap[0]}{cap[1]}"
    if arch not in set(torch.cuda.get_arch_list()):
        raise RuntimeError(
            f"PyTorch build does not contain {arch}. "
            "Restart Kaggle session with Internet ON."
        )

    x = torch.tensor([1.0, 2.0], device="cuda")
    y = (x * 2 + 1).sum()
    torch.cuda.synchronize()
    print("CUDA smoke test PASS:", float(y.cpu()))

print("Environment compatibility: PASS")


In [ ]:
# ============================================================
# 2. FIND DATASET + PRETRAIN CHECKPOINT
# ============================================================
KAGGLE_INPUT = Path("/kaggle/input")

dataset_roots = [
    p for p in KAGGLE_INPUT.rglob(DATASET_HINT)
    if p.is_dir()
]
if not dataset_roots:
    raise FileNotFoundError(
        f"Không tìm thấy dataset folder chứa '{DATASET_HINT}'. Add Input dataset trước."
    )
DATA_ROOT = sorted(dataset_roots, key=lambda p: len(p.parts))[0]

PROC_ROOT = DATA_ROOT / "processed/processed"
FRAME_ROOT = PROC_ROOT / "frame_splited"
KP_ROOT = PROC_ROOT / "keypoints_splited"

AUG_ROOT = DATA_ROOT / "processed_augmented/processed_augmented"
AUG_KP_ROOT = AUG_ROOT / "keypoints_splited"
AUG_FRAME_ROOT = AUG_ROOT / "frame_splited"

TRAIN_JSON = FRAME_ROOT / "train.json"
TEST_JSON = FRAME_ROOT / "test.json"
MANIFEST = FRAME_ROOT / "split_manifest.tsv"

required = [TRAIN_JSON, TEST_JSON, MANIFEST, KP_ROOT / "train", KP_ROOT / "test"]
for p in required:
    if not p.exists():
        raise FileNotFoundError(p)

pretrain_hits = sorted(KAGGLE_INPUT.rglob(PRETRAIN_NAME))
PRETRAIN_PATH = pretrain_hits[0] if pretrain_hits else None

print("DATA_ROOT:", DATA_ROOT)
print("KP_ROOT:", KP_ROOT)
print("FRAME_ROOT:", FRAME_ROOT)
print("AUG_KP_ROOT exists:", AUG_KP_ROOT.exists())
print("PRETRAIN:", PRETRAIN_PATH)

if REQUIRE_PRETRAIN and PRETRAIN_PATH is None:
    raise FileNotFoundError(
        f"Không tìm thấy {PRETRAIN_NAME}. "
        "Add Input output/dataset chứa model 3000+ từ cũ."
    )


In [ ]:
# ============================================================
# 3. BUILD EXACT 30-CLASS TABLE + BLACKLIST
# ============================================================
def norm_text(s):
    return " ".join(
        unicodedata.normalize("NFC", str(s)).strip().casefold().split()
    )

target_norm_to_name = {norm_text(x): x for x in TARGET_GLOSSES}
label_to_idx = {g:i for i,g in enumerate(TARGET_GLOSSES)}
idx_to_label = {i:g for g,i in label_to_idx.items()}

train_records = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
test_records = json.loads(TEST_JSON.read_text(encoding="utf-8"))

def canonical_rows(records, split):
    rows = []
    missing = []
    bad_removed = []

    for r in records:
        gnorm = norm_text(r["gloss"])
        if gnorm not in target_norm_to_name:
            continue

        gloss = target_norm_to_name[gnorm]
        vid = str(r["videoid"]).zfill(6)
        vid_int = int(vid)

        if vid_int in BAD_VIDEO_IDS.get(gloss, set()):
            bad_removed.append((split, gloss, vid))
            continue

        kp = KP_ROOT / split / gloss / f"{vid}.npy"
        if not kp.exists():
            # allow Unicode normalization/folder variation without global rglob
            folder = KP_ROOT / split
            matching = [
                d for d in folder.iterdir()
                if d.is_dir() and norm_text(d.name) == gnorm
            ]
            if matching:
                kp = matching[0] / f"{vid}.npy"

        if not kp.exists():
            missing.append((split, gloss, vid, str(kp)))
            continue

        rows.append({
            "videoid": vid,
            "gloss": gloss,
            "label": label_to_idx[gloss],
            "source_split": split,
            "kp_path": str(kp),
            "is_aug": False,
            "base_videoid": vid,
        })

    return pd.DataFrame(rows), bad_removed, missing

train_all_df, bad_train, missing_train = canonical_rows(train_records, "train")
test_df, bad_test, missing_test = canonical_rows(test_records, "test")

print("Canonical target train:", len(train_all_df))
print("Canonical target test :", len(test_df))
print("Blacklist removed     :", len(bad_train) + len(bad_test))
print("Missing keypoint files:", len(missing_train) + len(missing_test))

if missing_train[:5] or missing_test[:5]:
    print("Missing examples:", (missing_train + missing_test)[:5])

# Correct per-class count: records/videos, not rows of split_manifest.
count_table = (
    pd.concat([
        train_all_df.groupby("gloss").size().rename("train_after_blacklist"),
        test_df.groupby("gloss").size().rename("test_after_blacklist"),
    ], axis=1)
    .reindex(TARGET_GLOSSES)
    .fillna(0)
    .astype(int)
)
count_table["total_after_blacklist"] = (
    count_table["train_after_blacklist"] + count_table["test_after_blacklist"]
)

display(count_table)
count_table.to_csv(WORK_DIR / "counts_after_blacklist.csv", encoding="utf-8-sig")

missing_glosses = count_table[count_table["total_after_blacklist"] == 0].index.tolist()
if missing_glosses:
    raise RuntimeError(f"Không tìm thấy dữ liệu cho: {missing_glosses}")

low = count_table[count_table["total_after_blacklist"] < MIN_AFTER_FILTER_WARNING]
if len(low):
    print("WARNING — dưới", MIN_AFTER_FILTER_WARNING, "video sau blacklist:")
    display(low)

bad_df = pd.DataFrame(
    bad_train + bad_test, columns=["split","gloss","videoid"]
)
bad_df.to_csv(WORK_DIR / "removed_bad_video_ids.csv", index=False, encoding="utf-8-sig")


In [ ]:
# ============================================================
# 4. TRAIN / VAL / TEST SPLIT — signer-aware when metadata allows
# ============================================================
# Test is NEVER touched here.
# V4.3 tries signer metadata from:
#   1) split_manifest.tsv
#   2) train.json records
# If signer mapping is still unavailable, use deterministic per-gloss split.
# Robustness is then handled by a separate deterministic stress-validation view.

def _pick_manifest_col(columns, candidates):
    norm = {str(c).strip().casefold(): c for c in columns}
    for cand in candidates:
        if cand in norm:
            return norm[cand]
    for c in columns:
        cc = str(c).strip().casefold()
        if any(cand in cc for cand in candidates):
            return c
    return None

def _videoid_from_any(v):
    s = str(v)
    m = re.search(r"(?<!\d)(\d{6})(?!\d)", s)
    if m:
        return m.group(1)
    digits = re.sub(r"\D", "", s)
    return digits[-6:].zfill(6) if digits else None

SIGNER_KEYS = [
    "signer_id", "signer", "subject_id", "subject",
    "performer_id", "performer", "person_id", "person",
]

def _find_record_value(record, candidate_keys):
    """Case-insensitive lookup in top-level and one nested dict level."""
    flat = {}
    for k, v in record.items():
        flat[str(k).strip().casefold()] = v
        if isinstance(v, dict):
            for nk, nv in v.items():
                flat[str(nk).strip().casefold()] = nv
    for key in candidate_keys:
        if key in flat and flat[key] not in (None, ""):
            return flat[key]
    for k, v in flat.items():
        if any(key in k for key in candidate_keys) and v not in (None, ""):
            return v
    return None

def signer_map_from_json(records):
    out = {}
    for r in records:
        vid = _videoid_from_any(r.get("videoid", r.get("video_id", "")))
        signer = _find_record_value(r, SIGNER_KEYS)
        if vid is not None and signer not in (None, ""):
            out[vid] = str(signer).strip()
    return out

def fallback_stratified_split(df):
    rng = np.random.default_rng(SEED)
    train_parts, val_parts = [], []
    for gloss in TARGET_GLOSSES:
        cls = df[df.gloss == gloss].copy()
        idx = np.arange(len(cls))
        rng.shuffle(idx)
        n_val = max(3, int(round(len(cls) * VAL_FRACTION)))
        n_val = min(n_val, max(1, len(cls) - 2))
        val_parts.append(cls.iloc[idx[:n_val]])
        train_parts.append(cls.iloc[idx[n_val:]])
    return (
        pd.concat(train_parts, ignore_index=True),
        pd.concat(val_parts, ignore_index=True),
        "stratified_by_gloss_fallback",
        0.0,
        "none",
    )

train_df = val_df = None
VAL_SPLIT_METHOD = None
SIGNER_COVERAGE = 0.0
SIGNER_SOURCE = "none"

if TRY_SIGNER_AWARE_VAL:
    signer_maps = []

    # Source 1: TSV manifest
    try:
        manifest_df = pd.read_csv(MANIFEST, sep="\t")
        vid_col = _pick_manifest_col(
            manifest_df.columns,
            ["videoid", "video_id", "video", "clip_id", "clip"]
        )
        signer_col = _pick_manifest_col(manifest_df.columns, SIGNER_KEYS)

        print("Manifest columns:", list(manifest_df.columns))
        print("Detected video column:", vid_col, "| signer column:", signer_col)

        if vid_col is not None and signer_col is not None:
            mm = manifest_df[[vid_col, signer_col]].copy()
            mm["videoid"] = mm[vid_col].map(_videoid_from_any)
            mm["signer_group"] = mm[signer_col].astype(str).str.strip()
            mm = mm[(mm.videoid.notna()) & (mm.signer_group != "")]
            mm = (
                mm.groupby("videoid", as_index=False)["signer_group"]
                .agg(lambda x: x.value_counts().index[0])
            )
            signer_maps.append(("split_manifest.tsv", dict(zip(mm.videoid, mm.signer_group))))
    except Exception as e:
        print("Manifest signer extraction unavailable:", repr(e))

    # Source 2: train.json
    try:
        jm = signer_map_from_json(train_records)
        if jm:
            signer_maps.append(("train.json", jm))
            print("Signer-like entries found in train.json:", len(jm))
    except Exception as e:
        print("train.json signer extraction unavailable:", repr(e))

    # Choose the mapping source with best coverage on target canonical train.
    best_map, best_cov, best_source = None, 0.0, "none"
    for source_name, smap in signer_maps:
        cov = float(train_all_df.videoid.map(smap).notna().mean())
        print(f"Signer coverage from {source_name}: {cov:.1%}")
        if cov > best_cov:
            best_map, best_cov, best_source = smap, cov, source_name

    SIGNER_COVERAGE = best_cov
    SIGNER_SOURCE = best_source

    if best_map is not None and SIGNER_COVERAGE >= SIGNER_MAP_MIN_COVERAGE:
        tmp = train_all_df.copy()
        tmp["signer_group"] = tmp.videoid.map(best_map)
        # Unknown/online clips become independent groups; known clips stay signer-grouped.
        tmp["group_for_split"] = [
            s if pd.notna(s) else f"unknown_{v}"
            for s, v in zip(tmp.signer_group, tmp.videoid)
        ]

        n_groups = tmp.group_for_split.nunique()
        n_splits = min(max(3, int(round(1.0 / VAL_FRACTION))), n_groups)
        sgkf = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True, random_state=SEED
        )

        candidates = []
        X = np.zeros((len(tmp), 1), dtype=np.float32)
        y = tmp.label.to_numpy()
        groups = tmp.group_for_split.to_numpy()

        for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X, y, groups)):
            va = tmp.iloc[va_idx]
            all_labels = va.label.nunique() == len(TARGET_GLOSSES)
            frac_gap = abs(len(va) / len(tmp) - VAL_FRACTION)
            candidates.append((not all_labels, frac_gap, fold, tr_idx, va_idx))

        candidates.sort(key=lambda x: (x[0], x[1], x[2]))
        bad_all_labels, _, fold, tr_idx, va_idx = candidates[0]

        if not bad_all_labels:
            train_df = tmp.iloc[tr_idx].drop(
                columns=["signer_group", "group_for_split"]
            ).reset_index(drop=True)
            val_df = tmp.iloc[va_idx].drop(
                columns=["signer_group", "group_for_split"]
            ).reset_index(drop=True)
            VAL_SPLIT_METHOD = f"stratified_group_signer_fold_{fold}_of_{n_splits}"
            print("Using signer-aware validation:", VAL_SPLIT_METHOD)
            print("Signer source:", SIGNER_SOURCE)
        else:
            print("Signer split could not preserve all 30 labels -> fallback.")

if train_df is None or val_df is None:
    train_df, val_df, VAL_SPLIT_METHOD, SIGNER_COVERAGE, SIGNER_SOURCE = fallback_stratified_split(train_all_df)

train_ids_by_gloss = {
    g: set(train_df.loc[train_df.gloss == g, "videoid"])
    for g in TARGET_GLOSSES
}
val_ids_by_gloss = {
    g: set(val_df.loc[val_df.gloss == g, "videoid"])
    for g in TARGET_GLOSSES
}

print("Validation method:", VAL_SPLIT_METHOD)
print("Signer source    :", SIGNER_SOURCE)
print("Signer coverage  :", f"{SIGNER_COVERAGE:.1%}")
print("Canonical train  :", len(train_df))
print("Validation       :", len(val_df))
print("Final test       :", len(test_df))

assert set(train_df.videoid) & set(val_df.videoid) == set()
assert set(train_df.kp_path) & set(val_df.kp_path) == set()


In [ ]:
# ============================================================
# 5. ADD AUGMENTED KEYPOINT TRAIN FILES SAFELY
# ============================================================
def six_digit_id(text):
    m = re.search(r"(?<!\d)(\d{6})(?!\d)", str(text))
    return m.group(1) if m else None

aug_rows = []

if USE_AUGMENTED_KEYPOINTS and AUG_KP_ROOT.exists():
    for gloss in TARGET_GLOSSES:
        split_dir = AUG_KP_ROOT / "train"
        candidates = [
            d for d in split_dir.iterdir()
            if d.is_dir() and norm_text(d.name) == norm_text(gloss)
        ] if split_dir.exists() else []
        if not candidates:
            continue

        gdir = candidates[0]
        grouped = defaultdict(list)
        for p in gdir.glob("*.npy"):
            base = six_digit_id(p.stem)
            if base is None:
                continue
            if base in val_ids_by_gloss[gloss]:
                continue
            if base not in train_ids_by_gloss[gloss]:
                continue
            if int(base) in BAD_VIDEO_IDS.get(gloss, set()):
                continue
            grouped[base].append(p)

        for base, paths in grouped.items():
            paths = sorted(paths)
            # V3 always took lexicographically first variants. V4 samples a deterministic
            # subset so variant type is not biased by filename ordering.
            if len(paths) > MAX_AUG_PER_ORIGINAL:
                local_rng = np.random.default_rng(SEED + int(base))
                take = sorted(local_rng.choice(len(paths), MAX_AUG_PER_ORIGINAL, replace=False).tolist())
                paths = [paths[i] for i in take]

            for p in paths:
                aug_rows.append({
                    "videoid": p.stem,
                    "gloss": gloss,
                    "label": label_to_idx[gloss],
                    "source_split": "train_aug",
                    "kp_path": str(p),
                    "is_aug": True,
                    "base_videoid": base,
                })

aug_df = pd.DataFrame(aug_rows)
train_full_df = pd.concat([train_df, aug_df], ignore_index=True) if len(aug_df) else train_df.copy()

print("Augmented train samples accepted:", len(aug_df))
print("Total train samples:", len(train_full_df))

split_counts = pd.DataFrame({
    "train": train_full_df.groupby("gloss").size(),
    "val": val_df.groupby("gloss").size(),
    "test": test_df.groupby("gloss").size(),
}).reindex(TARGET_GLOSSES).fillna(0).astype(int)

display(split_counts)
split_counts.to_csv(WORK_DIR / "split_counts.csv", encoding="utf-8-sig")


In [ ]:
# ============================================================
# 6. KEYPOINT HEALTH CHECK — target files only
# ============================================================
def basic_kp_health(path):
    try:
        x = np.load(path, allow_pickle=False)
        if x.ndim != 3 or x.shape[-1] < 3 or x.shape[1] < KP_USED_POINTS:
            return False, f"shape={getattr(x,'shape',None)}"
        if x.shape[0] < 4:
            return False, f"too_short={x.shape[0]}"
        finite_ratio = np.isfinite(x[..., :3]).mean()
        if finite_ratio < 0.95:
            return False, f"finite={finite_ratio:.3f}"
        return True, "ok"
    except Exception as e:
        return False, repr(e)

health_bad = []

# Check every canonical target file + accepted aug; only a few thousand small npy files.
for row in tqdm(
    pd.concat([train_full_df, val_df, test_df], ignore_index=True).itertuples(index=False),
    total=len(train_full_df)+len(val_df)+len(test_df),
    desc="Keypoint health"
):
    ok, reason = basic_kp_health(row.kp_path)
    if not ok:
        health_bad.append({
            "videoid": row.videoid,
            "gloss": row.gloss,
            "source_split": row.source_split,
            "kp_path": row.kp_path,
            "reason": reason,
        })

health_bad_df = pd.DataFrame(health_bad)
print("Health-check failed:", len(health_bad_df))
if len(health_bad_df):
    display(health_bad_df.head(30))
    bad_paths = set(health_bad_df.kp_path)
    train_full_df = train_full_df[~train_full_df.kp_path.isin(bad_paths)].reset_index(drop=True)
    val_df = val_df[~val_df.kp_path.isin(bad_paths)].reset_index(drop=True)
    test_df = test_df[~test_df.kp_path.isin(bad_paths)].reset_index(drop=True)
    health_bad_df.to_csv(WORK_DIR / "removed_health_check.csv", index=False, encoding="utf-8-sig")

print("After health check:", len(train_full_df), len(val_df), len(test_df))


In [ ]:
# ============================================================
# 7. DATASET FORMAT DIAGNOSTIC
# ============================================================
sample_path = Path(train_df.iloc[0].kp_path)
sample = np.load(sample_path, allow_pickle=False)

print("Sample:", sample_path)
print("Shape:", sample.shape, "dtype:", sample.dtype)
print("Finite:", np.isfinite(sample).all())
print("Global min/max:", float(np.nanmin(sample)), float(np.nanmax(sample)))

if sample.shape[1] == 76:
    print(
        "Using points [0:75] as 33 pose + 21 left hand + 21 right hand. "
        "Dataset-specific point #76 is ignored for compatibility."
    )

# Estimate whether zero triples represent missing joints.
first75 = sample[:, :75, :3]
zero_ratio = float((np.linalg.norm(np.nan_to_num(first75), axis=-1) < 1e-8).mean())
print("Zero-joint ratio in sample:", zero_ratio)


In [ ]:
# ============================================================
# 8. PREPROCESSING + AUGMENTATION — reconstruction + conservative aug
# ============================================================
POSE_SLICE = slice(0, N_POSE)
LH_SLICE = slice(N_POSE, N_POSE + N_HAND)
RH_SLICE = slice(N_POSE + N_HAND, N_POSE + 2*N_HAND)
BODY_IDS_NP = np.array([0,11,12,13,14,15,16,23,24], dtype=np.int64)
BODY_IDS = torch.tensor(BODY_IDS_NP, dtype=torch.long)

# local-index skeleton edges after BODY_IDS selection
BODY_EDGES = [(1,2),(1,3),(3,5),(2,4),(4,6),(1,7),(2,8),(7,8)]
HAND_EDGES = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
]

def sanitize_raw76(arr):
    x = np.asarray(arr, dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x[:, :KP_USED_POINTS, :3]

def infer_mask(xyz):
    finite = np.isfinite(xyz).all(axis=-1)
    nonzero = np.linalg.norm(np.nan_to_num(xyz), axis=-1) > 1e-8
    return (finite & nonzero).astype(np.float32)[..., None]

def reconstruct_missing_keypoints(raw_xyz):
    """Linear interpolation between observations + np.interp edge padding.
    A joint needs at least RECON_MIN_OBS observed frames; otherwise it is left missing.
    """
    xyz = sanitize_raw76(raw_xyz)
    if not USE_KP_RECONSTRUCTION or len(xyz) < 3:
        return xyz

    mask = infer_mask(xyz)[..., 0] > 0.5
    out = xyz.copy()
    t_all = np.arange(len(out), dtype=np.float32)

    for j in range(out.shape[1]):
        idx = np.where(mask[:, j])[0]
        if len(idx) < RECON_MIN_OBS:
            continue
        for d in range(3):
            vals = out[idx, j, d]
            out[:, j, d] = np.interp(t_all, idx.astype(np.float32), vals).astype(np.float32)

    return np.nan_to_num(out).astype(np.float32)

def normalize_body(raw_xyz):
    xyz = reconstruct_missing_keypoints(raw_xyz)
    mask = infer_mask(xyz)

    shoulder_ok = (mask[:,11,0] > 0.5) & (mask[:,12,0] > 0.5)
    if shoulder_ok.any():
        centers = (xyz[shoulder_ok,11] + xyz[shoulder_ok,12]) / 2.0
        d = np.linalg.norm(xyz[shoulder_ok,11,:2] - xyz[shoulder_ok,12,:2], axis=1)
        good = np.isfinite(d) & (d >= MIN_SHOULDER_SCALE)
    else:
        good = np.zeros(0, dtype=bool)

    if shoulder_ok.any() and good.any():
        center = np.median(centers[good], axis=0)
        scale = float(np.median(d[good]))
        vec = (xyz[shoulder_ok,12,:2] - xyz[shoulder_ok,11,:2])[good]
        theta = -float(np.median(np.arctan2(vec[:,1], vec[:,0])))
    else:
        vis = mask[...,0] > 0.5
        pts = xyz[vis]
        if len(pts):
            center = np.median(pts, axis=0)
            radial = np.linalg.norm(pts[:,:2] - center[:2], axis=1)
            radial = radial[radial > 1e-4]
            scale = float(np.median(radial)) if len(radial) else 0.25
        else:
            center = np.zeros(3, np.float32)
            scale = 0.25
        theta = 0.0

    scale = max(scale, MIN_SHOULDER_SCALE)
    centers_t = np.repeat(center[None], len(xyz), axis=0)
    if shoulder_ok.any():
        centers_t[shoulder_ok] = (xyz[shoulder_ok,11] + xyz[shoulder_ok,12]) / 2.0

    out = (xyz - centers_t[:,None,:]) / scale
    c, s = np.cos(theta), np.sin(theta)
    R = np.array([[c,-s],[s,c]], dtype=np.float32)
    out[...,:2] = out[...,:2] @ R.T
    out = np.clip(out, -COORD_CLIP, COORD_CLIP) * mask
    return np.concatenate([out, mask], axis=-1).astype(np.float32)

def conservative_active_trim(x):
    if not USE_ACTIVE_TRIM or len(x) < 20:
        return x
    # wrists + fingertips; only trim a tiny edge fraction if motion strongly indicates idle frames
    ids = [15,16, 33+0,33+4,33+8,33+12,33+16,33+20, 54+0,54+4,54+8,54+12,54+16,54+20]
    pts = x[:, ids, :3]
    m = x[:, ids, 3:4]
    delta = np.zeros(len(x), np.float32)
    valid = m[1:] * m[:-1]
    dm = np.linalg.norm((pts[1:] - pts[:-1]) * valid, axis=-1)
    denom = np.maximum(valid[...,0].sum(1), 1.0)
    delta[1:] = dm.sum(1) / denom
    if not np.isfinite(delta).all() or delta.max() < 1e-5:
        return x
    smooth = np.convolve(delta, np.array([1,2,3,2,1], np.float32)/9.0, mode='same')
    positive = smooth[smooth > 0]
    if len(positive) < 5:
        return x
    threshold = max(float(np.quantile(positive, 0.55)), float(smooth.max()*0.12))
    active = np.where(smooth >= threshold)[0]
    if len(active) < 3:
        return x
    max_trim = max(1, int(round(len(x)*ACTIVE_TRIM_MAX_FRAC)))
    left = min(max(0, int(active[0])-2), max_trim)
    right = min(max(0, len(x)-1-int(active[-1])-2), max_trim)
    if left + right >= len(x)*0.20:
        return x
    return x[left:len(x)-right if right > 0 else len(x)]

def linear_resample(x, target_len):
    if len(x) == target_len:
        return x.astype(np.float32)
    if len(x) <= 1:
        return np.repeat(x[:1], target_len, axis=0).astype(np.float32)
    old_t = np.linspace(0, 1, len(x), dtype=np.float32)
    new_t = np.linspace(0, 1, target_len, dtype=np.float32)
    flat = x.reshape(len(x), -1)
    out = np.empty((target_len, flat.shape[1]), np.float32)
    for j in range(flat.shape[1]):
        out[:,j] = np.interp(new_t, old_t, flat[:,j])
    out = out.reshape(target_len, *x.shape[1:])
    out[...,3] = np.clip(out[...,3], 0, 1)
    out[...,:3] *= (out[...,3:4] > 0.2)
    return out.astype(np.float32)

def temporal_augment(x):
    T = len(x)
    if T >= 8 and random.random() < 0.65:
        keep = random.uniform(TEMP_CROP_MIN_KEEP, 1.0)
        n = max(6, int(round(T * keep)))
        start = random.randint(0, max(0, T-n))
        x = x[start:start+n]
    speed = random.uniform(TEMP_SPEED_MIN, TEMP_SPEED_MAX)
    speed_len = max(8, int(round(len(x) / speed)))
    return linear_resample(x, speed_len)

def spatial_augment(x):
    y = x.copy()
    xyz, mask = y[...,:3], y[...,3:4]
    angle = math.radians(random.uniform(-ROT_DEG, ROT_DEG))
    c, s = math.cos(angle), math.sin(angle)
    R = np.array([[c,-s],[s,c]], np.float32)
    xyz[...,:2] = xyz[...,:2] @ R.T
    xyz *= random.uniform(SCALE_MIN, SCALE_MAX)
    trans = np.random.normal(0, TRANS_STD, size=(1,1,3)).astype(np.float32)
    trans[...,2] *= 0.5
    xyz += trans
    if KP_NOISE_STD > 0:
        xyz += np.random.normal(0, KP_NOISE_STD, xyz.shape).astype(np.float32) * mask
    if POINT_DROP > 0:
        drop = (np.random.random(mask.shape[:-1]) < POINT_DROP)[...,None]
        mask = mask * (~drop)
    xyz = np.clip(xyz, -COORD_CLIP, COORD_CLIP) * mask
    y = np.concatenate([xyz, mask], axis=-1)
    if FRAME_DROP > 0 and len(y) > 4:
        dropf = np.random.random(len(y)) < FRAME_DROP
        y[dropf] = 0.0
    return y.astype(np.float32)

def _stable_stress_seed(key):
    s = str(key)
    return (SEED * 1000003 + sum((i + 1) * ord(ch) for i, ch in enumerate(s))) % (2**32 - 1)

def deterministic_stress_augment(x, key):
    """A fixed, mild perturbation for validation robustness scoring.
    It is deterministic per sample and does not use test data.
    """
    rng = np.random.default_rng(_stable_stress_seed(key))
    y = x.copy()

    if len(y) >= 8:
        keep = float(rng.uniform(STRESS_CROP_MIN_KEEP, 1.0))
        n = max(6, int(round(len(y) * keep)))
        start = int(rng.integers(0, max(1, len(y) - n + 1)))
        y = y[start:start+n]

    speed = float(rng.uniform(STRESS_SPEED_MIN, STRESS_SPEED_MAX))
    speed_len = max(8, int(round(len(y) / speed)))
    y = linear_resample(y, speed_len)
    y = linear_resample(y, SEQ_LEN)

    xyz = y[..., :3].copy()
    mask = y[..., 3:4].copy()

    angle = math.radians(float(rng.uniform(-STRESS_ROT_DEG, STRESS_ROT_DEG)))
    c, s = math.cos(angle), math.sin(angle)
    R = np.array([[c, -s], [s, c]], np.float32)
    xyz[..., :2] = xyz[..., :2] @ R.T
    xyz *= float(rng.uniform(STRESS_SCALE_MIN, STRESS_SCALE_MAX))

    trans = rng.normal(0, STRESS_TRANS_STD, size=(1, 1, 3)).astype(np.float32)
    trans[..., 2] *= 0.5
    xyz += trans

    if STRESS_NOISE_STD > 0:
        xyz += rng.normal(0, STRESS_NOISE_STD, xyz.shape).astype(np.float32) * mask

    if STRESS_POINT_DROP > 0:
        drop = (rng.random(mask.shape[:-1]) < STRESS_POINT_DROP)[..., None]
        mask = mask * (~drop)

    xyz = np.clip(xyz, -COORD_CLIP, COORD_CLIP) * mask
    y = np.concatenate([xyz, mask], axis=-1).astype(np.float32)

    if STRESS_FRAME_DROP > 0 and len(y) > 4:
        dropf = rng.random(len(y)) < STRESS_FRAME_DROP
        y[dropf] = 0.0

    return y.astype(np.float32)

def prepare_keypoints(path, train=False, is_aug=False, stress=False, stress_key=None):
    raw = np.load(path, allow_pickle=False)
    x = normalize_body(raw)
    x = conservative_active_trim(x)

    if train:
        # Canonical clips get full mild augmentation. Dataset-provided augmented clips
        # only get another perturbation occasionally to avoid destructive double-augmentation.
        extra = (not is_aug) or (random.random() < ALREADY_AUG_EXTRA_PROB)
        if extra:
            x = temporal_augment(x)
        x = linear_resample(x, SEQ_LEN)
        if extra:
            x = spatial_augment(x)
    elif stress:
        x = deterministic_stress_augment(x, stress_key if stress_key is not None else path)
    else:
        x = linear_resample(x, SEQ_LEN)

    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


In [ ]:
# ============================================================
# 9. RGB disabled in V3
# ============================================================
RGB_AVAILABLE = False
print("V3 mode: KEYPOINT-ONLY")
print("RGB_AVAILABLE:", RGB_AVAILABLE)


In [ ]:
# ============================================================
# 10. RGB disabled in V3
# ============================================================
def load_hand_rgb(*args, **kwargs):
    return None, None


In [ ]:
# ============================================================
# 11. KEYPOINT DATASET + MILD HARD-CLUSTER-AWARE P×K SAMPLER
# ============================================================
class VSL30Dataset(Dataset):
    def __init__(self, df, train=False, stress=False):
        self.df = df.reset_index(drop=True)
        self.train = bool(train)
        self.stress = bool(stress)
        if self.train and self.stress:
            raise ValueError("Dataset cannot be train=True and stress=True simultaneously.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        kp = prepare_keypoints(
            r.kp_path,
            train=self.train,
            is_aug=bool(getattr(r, "is_aug", False)),
            stress=self.stress,
            stress_key=str(r.videoid),
        )
        return {
            "kp": torch.from_numpy(kp),
            "label": torch.tensor(int(r.label), dtype=torch.long),
            "videoid": str(r.videoid),
            "gloss": str(r.gloss),
        }

class PKBatchSampler(Sampler):
    """P classes × K samples/class with mild class-selection weights.

    V4.3 avoids aggressive focus-pair oversampling. The hard cluster receives
    only 1.15–1.30x class-selection weights, while P×K still guarantees K
    positives for every selected class so SupCon / hard-negative metric loss
    has stable same-class positives.
    """
    def __init__(self, labels, p=10, k=4, batches=120, class_weights=None):
        self.labels = np.asarray(labels)
        self.p, self.k, self.batches = int(p), int(k), int(batches)
        self.by_class = {
            int(c): np.where(self.labels == c)[0]
            for c in np.unique(self.labels)
        }
        self.classes = np.asarray(sorted(self.by_class), dtype=int)
        class_weights = class_weights or {}
        self.weights = np.asarray(
            [float(class_weights.get(int(c), 1.0)) for c in self.classes],
            dtype=np.float64,
        )
        self.weights = np.clip(self.weights, 1e-6, None)

    def __len__(self):
        return self.batches

    def __iter__(self):
        rng = np.random.default_rng(SEED + random.randint(0, 10_000_000))
        n_pick = min(self.p, len(self.classes))

        for _ in range(self.batches):
            probs = self.weights / self.weights.sum()
            selected = rng.choice(
                self.classes, size=n_pick, replace=False, p=probs
            )

            batch = []
            for c in selected:
                inds = self.by_class[int(c)]
                chosen = rng.choice(
                    inds, size=self.k, replace=len(inds) < self.k
                )
                batch.extend(chosen.tolist())

            rng.shuffle(batch)
            yield batch

train_ds = VSL30Dataset(train_full_df, train=True)
val_ds = VSL30Dataset(val_df, train=False)
val_stress_ds = VSL30Dataset(val_df, train=False, stress=True)
test_ds = VSL30Dataset(test_df, train=False)

class_weights = {
    label_to_idx[g]: float(w)
    for g, w in HARD_CLASS_SAMPLING_BOOSTS.items()
    if g in label_to_idx
}

train_sampler = PKBatchSampler(
    train_full_df.label.values,
    p=PK_P,
    k=PK_K,
    batches=BATCHES_PER_EPOCH,
    class_weights=class_weights,
)

train_loader = DataLoader(
    train_ds, batch_sampler=train_sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=32, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_stress_loader = DataLoader(
    val_stress_ds, batch_size=32, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=32, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

b = next(iter(train_loader))
print("kp:", b["kp"].shape)
print("labels:", b["label"].shape, "unique:", b["label"].unique().numel())
print("Hard-class sampling weights:", HARD_CLASS_SAMPLING_BOOSTS)
print("Stress validation enabled:", USE_STRESS_VAL)


In [ ]:
# ============================================================
# 12. KEYPOINT ENCODER — old-compatible + bone/bone-motion residual + hand branch
# ============================================================
def first_order_motion(coords, mask):
    d = torch.zeros_like(coords)
    valid_pair = mask[:,1:] * mask[:,:-1]
    d[:,1:] = (coords[:,1:] - coords[:,:-1]) * valid_pair
    return d

def hand_local_features(hand):
    xyz = torch.nan_to_num(hand[..., :3], nan=0.0, posinf=0.0, neginf=0.0)
    mask = hand[..., 3:4].clamp(0.0,1.0)
    wrist = xyz[...,0:1,:]
    wrist_ok = mask[...,0:1,:] > 0.5
    mcp_ok = mask[...,9:10,:] > 0.5
    anchor_ok = wrist_ok & mcp_ok
    raw_scale = torch.linalg.norm(xyz[...,9:10,:] - wrist, dim=-1, keepdim=True)
    scale = torch.where(anchor_ok, raw_scale.clamp_min(MIN_HAND_SCALE), torch.ones_like(raw_scale))
    local = (xyz - wrist) / scale
    local = local * mask * wrist_ok.to(local.dtype)
    return torch.nan_to_num(local).clamp(-LOCAL_HAND_CLIP, LOCAL_HAND_CLIP)

def safe_angle(a,b,eps=1e-6):
    an = torch.linalg.norm(a,dim=-1).clamp_min(eps)
    bn = torch.linalg.norm(b,dim=-1).clamp_min(eps)
    cos = (a*b).sum(-1)/(an*bn)
    return torch.acos(torch.nan_to_num(cos).clamp(-0.9999,0.9999))

def hand_angle_descriptor(hand):
    xyz = torch.nan_to_num(hand[...,:3])
    mask = hand[...,3] > 0.5
    fingers = [[0,1,2,3,4],[0,5,6,7,8],[0,9,10,11,12],[0,13,14,15,16],[0,17,18,19,20]]
    feats = []
    for f in fingers:
        for j in range(1,4):
            a0,a1,a2 = f[j-1],f[j],f[j+1]
            ang = safe_angle(xyz[...,a0,:]-xyz[...,a1,:], xyz[...,a2,:]-xyz[...,a1,:])
            valid = mask[...,a0] & mask[...,a1] & mask[...,a2]
            feats.append(torch.where(valid,ang,torch.zeros_like(ang)))
    mcps = [1,5,9,13,17]
    rays = [xyz[...,i,:]-xyz[...,0,:] for i in mcps]
    for i in range(4):
        ang = safe_angle(rays[i],rays[i+1])
        valid = mask[...,0] & mask[...,mcps[i]] & mask[...,mcps[i+1]]
        feats.append(torch.where(valid,ang,torch.zeros_like(ang)))
    ang = safe_angle(rays[1],rays[4])
    valid = mask[...,0] & mask[...,mcps[1]] & mask[...,mcps[4]]
    feats.append(torch.where(valid,ang,torch.zeros_like(ang)))
    return torch.stack(feats,dim=-1)

def flatten_region_basic(region):
    xyz = region[...,:3]
    mask = region[...,3:4]
    motion = first_order_motion(xyz,mask)
    return torch.cat([(xyz*mask).flatten(-2), mask.flatten(-2), motion.flatten(-2)],dim=-1)

def flatten_bone_motion(region, edges):
    xyz = region[...,:3]
    mask = region[...,3:4]
    a = torch.tensor([e[0] for e in edges], device=region.device, dtype=torch.long)
    b = torch.tensor([e[1] for e in edges], device=region.device, dtype=torch.long)
    bone = xyz.index_select(2,b) - xyz.index_select(2,a)
    bm = mask.index_select(2,b) * mask.index_select(2,a)
    bone = bone * bm
    bone_motion = first_order_motion(bone, bm)
    return torch.cat([bone.flatten(-2), bm.flatten(-2), bone_motion.flatten(-2)], dim=-1)

class RegionMLP(nn.Module):
    def __init__(self,in_dim,out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim,out_dim), nn.LayerNorm(out_dim), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(out_dim,out_dim), nn.LayerNorm(out_dim),
        )
    def forward(self,x): return self.net(x)

class AttentionPool(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.score = nn.Linear(dim,1)
        nn.init.zeros_(self.score.weight); nn.init.zeros_(self.score.bias)
    def forward(self,x,mask=None):
        s = self.score(x).squeeze(-1)
        if mask is not None: s = s.masked_fill(~mask, -1e4)
        w = torch.softmax(s,dim=1)
        return (x*w.unsqueeze(-1)).sum(1), w

class KeypointEncoder30(nn.Module):
    def __init__(self):
        super().__init__()
        body_n = len(BODY_IDS_NP)
        body_in = body_n*(3+1+3)
        hand_in = N_HAND*(3+1) + N_HAND*3 + N_HAND*3 + (20 if USE_HAND_ANGLE_FEATURES else 0)

        # OLD-compatible modules
        self.body_encoder = RegionMLP(body_in,REGION_DIM)
        self.hand_encoder = RegionMLP(hand_in,REGION_DIM)
        self.side_embed = nn.Parameter(torch.zeros(2,REGION_DIM)); nn.init.normal_(self.side_embed,std=0.02)
        self.gate = nn.Sequential(nn.Linear(3*REGION_DIM,REGION_DIM),nn.GELU(),nn.Linear(REGION_DIM,3))
        self.fuse = nn.Sequential(nn.Linear(3*REGION_DIM,D_MODEL),nn.LayerNorm(D_MODEL),nn.GELU())
        self.local_conv = nn.Sequential(
            nn.Conv1d(D_MODEL,D_MODEL,3,padding=1,groups=D_MODEL),nn.GELU(),
            nn.Conv1d(D_MODEL,D_MODEL,1),nn.Dropout(DROPOUT),
        )
        self.pos = nn.Parameter(torch.zeros(1,SEQ_LEN,D_MODEL)); nn.init.trunc_normal_(self.pos,std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,nhead=NHEAD,dim_feedforward=FF_DIM,dropout=DROPOUT,
            activation="gelu",batch_first=True,norm_first=True,
        )
        self.temporal = nn.TransformerEncoder(layer,num_layers=NUM_TRANSFORMER_LAYERS)
        self.attn_pool = AttentionPool(D_MODEL)
        self.head = nn.Sequential(
            nn.LayerNorm(D_MODEL),nn.Linear(D_MODEL,D_MODEL),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(D_MODEL,EMBED_DIM)
        )

        # NEW geometry residual: bone + bone-motion
        self.body_geom_encoder = RegionMLP(len(BODY_EDGES)*7, REGION_DIM)
        self.hand_geom_encoder = RegionMLP(len(HAND_EDGES)*7, REGION_DIM)
        self.body_geom_alpha = nn.Parameter(torch.tensor(-1.5))
        self.hand_geom_alpha = nn.Parameter(torch.tensor(-1.5))

        # NEW hand-only temporal representation
        self.hand_fuse = nn.Sequential(nn.Linear(2*REGION_DIM,HAND_TEMP_DIM),nn.LayerNorm(HAND_TEMP_DIM),nn.GELU())
        self.hand_local_conv = nn.Sequential(
            nn.Conv1d(HAND_TEMP_DIM,HAND_TEMP_DIM,3,padding=1,groups=HAND_TEMP_DIM),nn.GELU(),
            nn.Conv1d(HAND_TEMP_DIM,HAND_TEMP_DIM,1),nn.Dropout(DROPOUT),
        )
        self.hand_pool = AttentionPool(HAND_TEMP_DIM)
        self.hand_head = nn.Sequential(
            nn.LayerNorm(HAND_TEMP_DIM),nn.Linear(HAND_TEMP_DIM,EMBED_DIM),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(EMBED_DIM,EMBED_DIM)
        )

    def encode_hand_base(self,hand,side):
        xyz, mask = hand[...,:3], hand[...,3:4]
        local = hand_local_features(hand)
        motion = first_order_motion(xyz,mask)
        parts = [(xyz*mask).flatten(-2),mask.flatten(-2),local.flatten(-2),motion.flatten(-2)]
        if USE_HAND_ANGLE_FEATURES: parts.append(hand_angle_descriptor(hand))
        h = self.hand_encoder(torch.cat(parts,dim=-1))
        return h + self.side_embed[side][None,None,:]

    def forward(self,x,return_attention=False,return_parts=False):
        x = torch.nan_to_num(x.float())
        xyz = x[...,:3].clamp(-COORD_CLIP,COORD_CLIP)
        mask = x[...,3:4].clamp(0,1)
        x = torch.cat([xyz,mask],dim=-1)

        body = x.index_select(2,BODY_IDS.to(x.device))
        lh, rh = x[:,:,LH_SLICE,:], x[:,:,RH_SLICE,:]

        rb = self.body_encoder(flatten_region_basic(body))
        rl = self.encode_hand_base(lh,0)
        rr = self.encode_hand_base(rh,1)

        if USE_GEOMETRY_RESIDUAL:
            ab = torch.sigmoid(self.body_geom_alpha)
            ah = torch.sigmoid(self.hand_geom_alpha)
            rb = rb + ab * self.body_geom_encoder(flatten_bone_motion(body, BODY_EDGES))
            rl = rl + ah * self.hand_geom_encoder(flatten_bone_motion(lh, HAND_EDGES))
            rr = rr + ah * self.hand_geom_encoder(flatten_bone_motion(rh, HAND_EDGES))

        regions = [rb,rl,rr]
        cat = torch.cat(regions,dim=-1)
        gates = torch.softmax(self.gate(cat),dim=-1)
        weighted = [r*gates[...,i:i+1] for i,r in enumerate(regions)]

        h = self.fuse(torch.cat(weighted,dim=-1))
        h = h + self.local_conv(h.transpose(1,2)).transpose(1,2)
        h = h + self.pos[:,:h.size(1)]
        h = self.temporal(h)
        frame_valid = (mask.sum(dim=(2,3)) > 0)
        pooled, attn = self.attn_pool(h,frame_valid)
        z = F.normalize(torch.nan_to_num(self.head(pooled)).float(),dim=-1,eps=1e-6)

        hand_z = hand_attn = None
        if return_parts:
            hh = self.hand_fuse(torch.cat([rl,rr],dim=-1))
            hh = hh + self.hand_local_conv(hh.transpose(1,2)).transpose(1,2)
            hand_pooled, hand_attn = self.hand_pool(hh, frame_valid)
            hand_z = F.normalize(torch.nan_to_num(self.hand_head(hand_pooled)).float(),dim=-1,eps=1e-6)

        if return_attention and return_parts:
            return z, hand_z, attn, hand_attn, gates
        if return_attention:
            return z, attn, gates
        if return_parts:
            return z, hand_z
        return z


In [ ]:
# ============================================================
# 13. LOAD COMPATIBLE WEIGHTS FROM OLD 3000+ WORD MODEL
# ============================================================
kp_encoder = KeypointEncoder30()
loaded_report, skipped_report = [], []

if PRETRAIN_PATH is not None:
    old = torch.load(PRETRAIN_PATH, map_location="cpu", weights_only=False)
    old_sd = old.get("model_state", old)
    new_sd = kp_encoder.state_dict()
    transfer_prefixes = (
        "body_encoder.", "hand_encoder.", "side_embed", "local_conv.",
        "pos", "temporal.", "head.",
    )
    for k,v in old_sd.items():
        if not k.startswith(transfer_prefixes):
            continue
        if k in new_sd and tuple(new_sd[k].shape) == tuple(v.shape):
            new_sd[k] = v
            loaded_report.append(k)
        else:
            skipped_report.append(k)
    kp_encoder.load_state_dict(new_sd)

print("Transferred old tensors:", len(loaded_report))
print("Skipped incompatible tensors:", len(skipped_report))
print("New V4 modules start fresh: geometry residual + hand-only branch")

if REQUIRE_PRETRAIN and len(loaded_report) < 20:
    raise RuntimeError("Checkpoint found but too few tensors are compatible; verify best_vsl_metric_encoder.pt")

kp_encoder = kp_encoder.to(DEVICE)
with torch.no_grad():
    smoke = b["kp"][:2].to(DEVICE)
    z,hand_z,attn,hand_attn,gates = kp_encoder(smoke,return_attention=True,return_parts=True)
print("Global embedding:", z.shape, "| hand embedding:", hand_z.shape)
print("Attention:", attn.shape, "| hand attention:", hand_attn.shape, "| gates:", gates.shape)
print("Finite:", bool(torch.isfinite(z).all() and torch.isfinite(hand_z).all()))


In [ ]:
# ============================================================
# 14. RGB remains disabled
# ============================================================
RGB_AVAILABLE = False
print("V4.3 mode: KEYPOINT-ONLY; RGB branch disabled.")


In [ ]:
# ============================================================
# 15. V4.3 CLASSIFIER — main + cosine metric + hand auxiliary
# ============================================================
class CosineClassifier(nn.Module):
    def __init__(self, dim, n_classes, scale=12.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_classes,dim))
        self.scale = float(scale)
        nn.init.xavier_uniform_(self.weight)
    def forward(self,z):
        z=F.normalize(z.float(),dim=-1)
        w=F.normalize(self.weight.float(),dim=-1)
        return self.scale*(z@w.T)

class VSL30Model(nn.Module):
    def __init__(self,kp_encoder):
        super().__init__()
        self.kp_encoder=kp_encoder
        self.classifier=nn.Sequential(nn.LayerNorm(EMBED_DIM),nn.Dropout(0.10),nn.Linear(EMBED_DIM,len(TARGET_GLOSSES)))
        self.kp_classifier=nn.Linear(EMBED_DIM,len(TARGET_GLOSSES))
        self.metric_classifier=CosineClassifier(EMBED_DIM,len(TARGET_GLOSSES),METRIC_SCALE)
        self.hand_classifier=nn.Sequential(nn.LayerNorm(EMBED_DIM),nn.Dropout(0.08),nn.Linear(EMBED_DIM,len(TARGET_GLOSSES)))
    def forward(self,kp):
        z,hand_z=self.kp_encoder(kp,return_parts=True)
        return {
            "logits":self.classifier(z),
            "kp_logits":self.kp_classifier(z),
            "metric_logits":self.metric_classifier(z),
            "hand_logits":self.hand_classifier(hand_z),
            "embedding":z,
            "hand_embedding":hand_z,
        }

model=VSL30Model(kp_encoder).to(DEVICE)
params=sum(p.numel() for p in model.parameters())
print(f"Total params: {params/1e6:.2f}M")
with torch.no_grad():
    out_smoke=model(b["kp"][:2].to(DEVICE))
print({k:tuple(v.shape) for k,v in out_smoke.items() if hasattr(v,'shape')})


In [ ]:
# ============================================================
# 16. V4.3 LOSSES — local hard-negative metric learning
# ============================================================
def supervised_contrastive_loss(z, labels, temp=0.08):
    z = F.normalize(z.float(), dim=-1)
    labels = labels.view(-1, 1)

    sim = (z @ z.T) / temp
    eye = torch.eye(len(z), device=z.device, dtype=torch.bool)
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()

    exp_sim = torch.exp(sim) * (~eye)
    pos = (labels == labels.T) & (~eye)
    denom = exp_sim.sum(dim=1).clamp_min(1e-12)
    pos_count = pos.sum(dim=1)
    valid = pos_count > 0

    log_prob = sim - torch.log(denom).unsqueeze(1)
    if not valid.any():
        return z.sum() * 0.0

    return (
        -(log_prob * pos).sum(dim=1)[valid] / pos_count[valid].float()
    ).mean()

def additive_metric_margin_logits(metric_logits, labels):
    out = metric_logits.clone()
    out[
        torch.arange(len(labels), device=labels.device),
        labels,
    ] -= METRIC_SCALE * METRIC_MARGIN
    return out

def focus_pair_ce_loss(logits, labels):
    """Small binary boundary term for Thích-vs-Muốn only.

    Weight is intentionally lower than V4.2. Hard-negative embedding loss is
    responsible for broader local separation from Anh/Bố/Ăn instead.
    """
    a, b = FOCUS_PAIR
    ia, ib = label_to_idx[a], label_to_idx[b]
    m = (labels == ia) | (labels == ib)

    if not m.any():
        return logits.sum() * 0.0

    pair_logits = logits[m][:, [ia, ib]].float()
    pair_targets = (labels[m] == ib).long()
    return F.cross_entropy(pair_logits, pair_targets)

def pair_margin_loss(logits, labels, pair, margin):
    a, b = pair
    ia, ib = label_to_idx[a], label_to_idx[b]
    terms = []

    for true_idx, other_idx in [(ia, ib), (ib, ia)]:
        m = labels == true_idx
        if m.any():
            gap = logits[m, true_idx] - logits[m, other_idx]
            terms.append(F.relu(margin - gap).mean())

    return torch.stack(terms).mean() if terms else logits.sum() * 0.0

def hard_negative_metric_loss(z, labels):
    """Targeted hardest-negative loss on normalized global embeddings.

    For each hard-cluster anchor:
      mean(same-class cosine similarity) >= mean(top-k mapped hard negatives) + margin

    This is local: easy classes are not forced apart more than ordinary SupCon
    already does. If mapped negative classes are absent from the P×K batch,
    that anchor is skipped instead of fabricating a negative.
    """
    z = F.normalize(z.float(), dim=-1)
    sim = z @ z.T
    n = len(z)
    eye = torch.eye(n, device=z.device, dtype=torch.bool)
    terms = []

    for gloss, neg_glosses in HARD_NEGATIVE_MAP.items():
        if gloss not in label_to_idx:
            continue
        anchor_cls = label_to_idx[gloss]
        neg_ids = [label_to_idx[g] for g in neg_glosses if g in label_to_idx]
        if not neg_ids:
            continue

        anchor_idx = torch.where(labels == anchor_cls)[0]
        for i in anchor_idx:
            pos_mask = (labels == anchor_cls) & (~eye[i])
            neg_mask = torch.zeros_like(labels, dtype=torch.bool)
            for nid in neg_ids:
                neg_mask |= labels == nid

            if not pos_mask.any() or not neg_mask.any():
                continue

            pos_sim = sim[i][pos_mask].mean()
            neg_sim_all = sim[i][neg_mask]
            k = min(int(HARD_NEG_TOPK), int(neg_sim_all.numel()))
            hard_neg_sim = torch.topk(neg_sim_all, k=k).values.mean()
            terms.append(F.relu(HARD_NEG_MARGIN + hard_neg_sim - pos_sim))

    return torch.stack(terms).mean() if terms else z.sum() * 0.0

def compute_loss(outputs, labels):
    main_logits = outputs["logits"].float()

    main_ce = F.cross_entropy(
        main_logits, labels, label_smoothing=LABEL_SMOOTHING
    )
    kp_ce = F.cross_entropy(outputs["kp_logits"].float(), labels)
    hand_ce = F.cross_entropy(outputs["hand_logits"].float(), labels)
    metric_ce = F.cross_entropy(
        additive_metric_margin_logits(
            outputs["metric_logits"].float(), labels
        ),
        labels,
    )

    sup = supervised_contrastive_loss(
        outputs["embedding"], labels, SUPCON_TEMP
    )
    hand_sup = supervised_contrastive_loss(
        outputs["hand_embedding"], labels, SUPCON_TEMP
    )

    focus_ce = focus_pair_ce_loss(main_logits, labels)
    focus_margin = pair_margin_loss(
        main_logits, labels, FOCUS_PAIR, FOCUS_PAIR_MARGIN
    )
    hard_neg = hard_negative_metric_loss(outputs["embedding"], labels)

    total = (
        main_ce
        + KEYPOINT_AUX_WEIGHT * kp_ce
        + HAND_AUX_WEIGHT * hand_ce
        + METRIC_CE_WEIGHT * metric_ce
        + SUPCON_WEIGHT * sup
        + HAND_SUPCON_WEIGHT * hand_sup
        + FOCUS_PAIR_CE_WEIGHT * focus_ce
        + FOCUS_PAIR_MARGIN_WEIGHT * focus_margin
        + HARD_NEG_METRIC_WEIGHT * hard_neg
    )

    return total, {
        "main_ce": float(main_ce.detach()),
        "kp_ce": float(kp_ce.detach()),
        "hand_ce": float(hand_ce.detach()),
        "metric_ce": float(metric_ce.detach()),
        "supcon": float(sup.detach()),
        "hand_supcon": float(hand_sup.detach()),
        "focus_pair_ce": float(focus_ce.detach()),
        "focus_pair_margin": float(focus_margin.detach()),
        "hard_negative_metric": float(hard_neg.detach()),
    }


In [ ]:
# ============================================================
# 17. V4.3 TRAINING STAGES / OPTIMIZER
# ============================================================
def set_stage(stage):
    for p in model.parameters():
        p.requires_grad = False

    # New heads + geometry/hand auxiliary modules are always trainable.
    always_modules = [
        model.classifier,
        model.kp_classifier,
        model.metric_classifier,
        model.hand_classifier,
        model.kp_encoder.gate,
        model.kp_encoder.fuse,
        model.kp_encoder.attn_pool,
        model.kp_encoder.body_geom_encoder,
        model.kp_encoder.hand_geom_encoder,
        model.kp_encoder.hand_fuse,
        model.kp_encoder.hand_local_conv,
        model.kp_encoder.hand_pool,
        model.kp_encoder.hand_head,
    ]
    for module in always_modules:
        for p in module.parameters():
            p.requires_grad = True

    model.kp_encoder.body_geom_alpha.requires_grad = True
    model.kp_encoder.hand_geom_alpha.requires_grad = True

    # Stage 2 unfreezes temporal adaptation, but keeps low-level pretrained
    # body/hand encoders frozen. This remains the best adaptation region from V4.2.
    if stage >= 2:
        for module in [
            model.kp_encoder.local_conv,
            model.kp_encoder.temporal,
            model.kp_encoder.head,
        ]:
            for p in module.parameters():
                p.requires_grad = True
        model.kp_encoder.pos.requires_grad = True

OLD_NAMES = (
    "kp_encoder.local_conv",
    "kp_encoder.temporal",
    "kp_encoder.head",
    "kp_encoder.pos",
)

def build_optimizer(stage):
    old_params, new_params = [], []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (old_params if name.startswith(OLD_NAMES) else new_params).append(p)

    if stage == 1:
        groups = [{"params": new_params, "lr": LR_NEW_STAGE1}]
    elif stage == 2:
        groups = [
            {"params": old_params, "lr": LR_OLD_STAGE2},
            {"params": new_params, "lr": LR_NEW_STAGE2},
        ]
    else:
        raise ValueError("V4.3 only uses Stage 1 and Stage 2.")

    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

for s in [1, 2]:
    set_stage(s)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Stage {s}: trainable {n/1e6:.2f}M")

print("Stage 3: DISABLED in V4.3")


In [ ]:
# ============================================================
# 18. EVALUATION — MAIN HEAD ONLY + HARD-CLUSTER METRICS
# ============================================================
def _hard_class_scores(y, pred):
    per_f1 = f1_score(
        y, pred,
        labels=np.arange(len(TARGET_GLOSSES)),
        average=None,
        zero_division=0,
    )
    ids = [label_to_idx[g] for g in HARD_CLUSTER if g in label_to_idx]
    vals = np.asarray([per_f1[i] for i in ids], dtype=np.float64)
    return {
        "hard_macro_f1": float(vals.mean()) if len(vals) else 0.0,
        "hard_min_f1": float(vals.min()) if len(vals) else 0.0,
    }

def metrics_from_probs(y, p):
    pred = p.argmax(1)
    out = {
        "acc": accuracy_score(y, pred),
        "balanced_acc": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
        "top3": top_k_accuracy_score(
            y, p, k=min(3, p.shape[1]),
            labels=np.arange(p.shape[1]),
        ),
    }
    out.update(_hard_class_scores(y, pred))
    return out

@torch.no_grad()
def collect_main_probs(loader):
    model.eval()
    ys, vids, probs = [], [], []

    for batch in loader:
        kp = batch["kp"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE)

        o = model(kp)
        p = torch.softmax(o["logits"].float(), -1)

        ys.append(y.cpu().numpy())
        vids.extend(batch["videoid"])
        probs.append(p.cpu().numpy())

    return np.concatenate(ys), np.concatenate(probs), vids

@torch.no_grad()
def evaluate(loader, return_predictions=False):
    y, p, vids = collect_main_probs(loader)
    pred = p.argmax(1)
    metrics = metrics_from_probs(y, p)

    if not return_predictions:
        return metrics

    order = np.argsort(-p, axis=1)
    rows = []

    for i in range(len(y)):
        top1, top2 = int(order[i, 0]), int(order[i, 1])
        rows.append({
            "videoid": vids[i],
            "true": idx_to_label[int(y[i])],
            "pred": idx_to_label[int(pred[i])],
            "correct": int(pred[i] == y[i]),
            "confidence": float(p[i, pred[i]]),
            "top1_top2_margin": float(p[i, top1] - p[i, top2]),
            "top3": " | ".join(
                idx_to_label[int(j)] for j in order[i, :3]
            ),
            "top3_scores": " | ".join(
                f"{p[i, j]:.4f}" for j in order[i, :3]
            ),
        })

    return metrics, pd.DataFrame(rows), y, pred, p

def selection_score(clean_metrics, stress_metrics):
    """V4.3 checkpoint score.

    Overall clean Macro-F1 remains dominant. Hard-cluster mean/min F1 prevent
    a checkpoint that gains on easy classes while sacrificing Thích/Muốn/Ăn/Bố/Anh.
    """
    stress_macro = (
        float(stress_metrics["macro_f1"])
        if USE_STRESS_VAL else float(clean_metrics["macro_f1"])
    )
    return float(
        SELECT_CLEAN_MACRO_W * clean_metrics["macro_f1"]
        + SELECT_STRESS_MACRO_W * stress_macro
        + SELECT_HARD_MEAN_W * clean_metrics["hard_macro_f1"]
        + SELECT_HARD_MIN_W * clean_metrics["hard_min_f1"]
    )


In [ ]:
# ============================================================
# 19. FULL 30-CLASS TRAIN — V4.3 ROBUST + HARD-CLASS CHECKPOINTING
# ============================================================
history = []
best_selection_score = -1.0
best_clean_f1 = -1.0
best_hard_f1 = -1.0
best_hard_min = -1.0
best_clean_acc = -1.0

best_path = WORK_DIR / "best_vsl30_v4_3.pt"
global_epoch = 0
no_improve = 0

stage_plan = [
    (1, STAGE1_EPOCHS),
    (2, STAGE2_EPOCHS),
]

amp_enabled = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
stop_all = False

for stage, n_epochs in stage_plan:
    if stop_all:
        break

    set_stage(stage)
    optimizer = build_optimizer(stage)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, n_epochs)
    )

    print()
    print("=" * 80)
    print(f"STAGE {stage} — {n_epochs} epochs")
    print("=" * 80)

    no_improve = 0

    for _ in range(n_epochs):
        global_epoch += 1
        model.train()

        running = 0.0
        steps = 0
        pbar = tqdm(train_loader, desc=f"E{global_epoch:02d} S{stage}")

        for batch in pbar:
            kp = batch["kp"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                "cuda",
                enabled=amp_enabled,
                dtype=torch.float16,
            ):
                outputs = model(kp)

            loss, parts = compute_loss(outputs, y)

            if not torch.isfinite(loss):
                print("Non-finite loss skipped:", parts)
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )

            scaler.step(optimizer)
            scaler.update()

            running += float(loss.detach())
            steps += 1
            pbar.set_postfix(
                loss=f"{running/max(1, steps):.4f}",
                hardneg=f"{parts['hard_negative_metric']:.3f}",
            )

        scheduler.step()

        clean_val = evaluate(val_loader)
        stress_val = evaluate(val_stress_loader) if USE_STRESS_VAL else clean_val
        sel_score = selection_score(clean_val, stress_val)
        train_loss = running / max(1, steps)

        row = {
            "epoch": global_epoch,
            "stage": stage,
            "train_loss": train_loss,
            "selection_score": sel_score,
            **{f"val_{k}": v for k, v in clean_val.items()},
            **{f"stress_{k}": v for k, v in stress_val.items()},
        }
        history.append(row)

        print(
            f"E{global_epoch:02d} loss={train_loss:.4f} | "
            f"val_acc={clean_val['acc']:.4f} | "
            f"val_macroF1={clean_val['macro_f1']:.4f} | "
            f"hard_mean={clean_val['hard_macro_f1']:.4f} | "
            f"hard_min={clean_val['hard_min_f1']:.4f} | "
            f"stress_macroF1={stress_val['macro_f1']:.4f} | "
            f"select={sel_score:.4f} | "
            f"top3={clean_val['top3']:.4f}"
        )

        clean_f1 = float(clean_val["macro_f1"])
        hard_f1 = float(clean_val["hard_macro_f1"])
        hard_min = float(clean_val["hard_min_f1"])
        clean_acc = float(clean_val["acc"])

        is_better = (
            sel_score > best_selection_score + CHECKPOINT_EPS
            or (
                abs(sel_score - best_selection_score) <= CHECKPOINT_EPS
                and hard_min > best_hard_min + CHECKPOINT_EPS
            )
            or (
                abs(sel_score - best_selection_score) <= CHECKPOINT_EPS
                and abs(hard_min - best_hard_min) <= CHECKPOINT_EPS
                and clean_f1 > best_clean_f1 + CHECKPOINT_EPS
            )
            or (
                abs(sel_score - best_selection_score) <= CHECKPOINT_EPS
                and abs(hard_min - best_hard_min) <= CHECKPOINT_EPS
                and abs(clean_f1 - best_clean_f1) <= CHECKPOINT_EPS
                and clean_acc > best_clean_acc + CHECKPOINT_EPS
            )
        )

        if is_better:
            best_selection_score = sel_score
            best_clean_f1 = clean_f1
            best_hard_f1 = hard_f1
            best_hard_min = hard_min
            best_clean_acc = clean_acc
            no_improve = 0

            torch.save({
                "model_state": model.state_dict(),
                "epoch": global_epoch,
                "stage": stage,
                "best_selection_score": best_selection_score,
                "best_val_macro_f1": best_clean_f1,
                "best_val_hard_macro_f1": best_hard_f1,
                "best_val_hard_min_f1": best_hard_min,
                "best_val_acc": best_clean_acc,
                "val_metrics": clean_val,
                "stress_val_metrics": stress_val,
                "target_glosses": TARGET_GLOSSES,
                "config": {
                    "seq_len": SEQ_LEN,
                    "kp_used_points": KP_USED_POINTS,
                    "mode": "keypoint_only_v4_3",
                    "embed_dim": EMBED_DIM,
                    "d_model": D_MODEL,
                    "pretrain_source": str(PRETRAIN_PATH),
                    "val_split_method": VAL_SPLIT_METHOD,
                    "signer_coverage": SIGNER_COVERAGE,
                    "signer_source": SIGNER_SOURCE,
                    "training_scope": "30_target_glosses_only",
                    "kp_reconstruction": USE_KP_RECONSTRUCTION,
                    "bone_motion_residual": USE_GEOMETRY_RESIDUAL,
                    "focus_pair": FOCUS_PAIR,
                    "hard_cluster": HARD_CLUSTER,
                    "hard_negative_map": HARD_NEGATIVE_MAP,
                    "hard_negative_metric_weight": HARD_NEG_METRIC_WEIGHT,
                    "hard_negative_margin": HARD_NEG_MARGIN,
                    "stage3_enabled": False,
                    "prediction_head": "main_only",
                    "selection_weights": {
                        "clean_macro": SELECT_CLEAN_MACRO_W,
                        "stress_macro": SELECT_STRESS_MACRO_W,
                        "hard_mean": SELECT_HARD_MEAN_W,
                        "hard_min": SELECT_HARD_MIN_W,
                    },
                },
            }, best_path)

            print("  ↳ SAVED BEST:", best_path.name)
        else:
            no_improve += 1

        pd.DataFrame(history).to_csv(
            WORK_DIR / "training_history.csv", index=False
        )

        if stage == 2 and no_improve >= EARLY_STOP_PATIENCE:
            print("Early stopping Stage 2.")
            stop_all = True
            break

print()
print("BEST SELECTION SCORE:", best_selection_score)
print("BEST CLEAN VAL MACRO F1:", best_clean_f1)
print("BEST HARD-CLUSTER MEAN F1:", best_hard_f1)
print("BEST HARD-CLUSTER MIN F1:", best_hard_min)
print("BEST CLEAN VAL ACC:", best_clean_acc)
print("BEST CHECKPOINT:", best_path)


In [ ]:
# ============================================================
# 20. LOAD BEST V4.3 CHECKPOINT — MAIN HEAD ONLY — TEST ONCE
# ============================================================
best = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model_state"], strict=True)

val_metrics, val_pred_df, val_y, val_pred, val_probs = evaluate(
    val_loader, return_predictions=True
)
stress_val_metrics, stress_val_pred_df, _, _, _ = evaluate(
    val_stress_loader, return_predictions=True
) if USE_STRESS_VAL else (
    val_metrics, val_pred_df.copy(), val_y, val_pred, val_probs
)

# No metric/hand fusion. No test-time threshold tuning.
test_metrics, test_pred_df, y_true, y_pred, test_probs = evaluate(
    test_loader, return_predictions=True
)

print("BEST EPOCH:", best.get("epoch"), "| STAGE:", best.get("stage"))
print("VAL CLEAN :", json.dumps(val_metrics, indent=2, ensure_ascii=False))
print("VAL STRESS:", json.dumps(stress_val_metrics, indent=2, ensure_ascii=False))
print("TEST MAIN :", json.dumps(test_metrics, indent=2, ensure_ascii=False))

display(test_pred_df.head(30))

val_pred_df.to_csv(
    WORK_DIR / "validation_predictions_clean.csv",
    index=False, encoding="utf-8-sig",
)
stress_val_pred_df.to_csv(
    WORK_DIR / "validation_predictions_stress.csv",
    index=False, encoding="utf-8-sig",
)
test_pred_df.to_csv(
    WORK_DIR / "test_predictions.csv",
    index=False, encoding="utf-8-sig",
)

report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(TARGET_GLOSSES)),
    target_names=TARGET_GLOSSES,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).T.to_csv(
    WORK_DIR / "classification_report.csv",
    encoding="utf-8-sig",
)


In [ ]:
# ============================================================
# 21. CONFUSION MATRIX + FOCUS/HARD-CLUSTER DIAGNOSTICS
# ============================================================
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(len(TARGET_GLOSSES)),
)

cm_df = pd.DataFrame(
    cm,
    index=TARGET_GLOSSES,
    columns=TARGET_GLOSSES,
)
cm_df.to_csv(
    WORK_DIR / "confusion_matrix.csv",
    encoding="utf-8-sig",
)

plt.figure(figsize=(15, 13))
plt.imshow(cm)
plt.xticks(
    range(len(TARGET_GLOSSES)),
    TARGET_GLOSSES,
    rotation=90,
    fontsize=8,
)
plt.yticks(
    range(len(TARGET_GLOSSES)),
    TARGET_GLOSSES,
    fontsize=8,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("VSL30 V4.3 confusion matrix — main head only")
plt.colorbar()
plt.tight_layout()
plt.savefig(
    WORK_DIR / "confusion_matrix.png",
    dpi=180,
)
plt.show()

pairs = []
for i in range(len(TARGET_GLOSSES)):
    row = cm[i].copy()
    row[i] = 0
    j = int(row.argmax())

    if row[j] > 0:
        pairs.append({
            "true": TARGET_GLOSSES[i],
            "most_confused_with": TARGET_GLOSSES[j],
            "count": int(row[j]),
            "class_total": int(cm[i].sum()),
            "error_rate_to_pair": float(
                row[j] / max(1, cm[i].sum())
            ),
        })

hard_pairs_df = pd.DataFrame(pairs).sort_values(
    ["count", "error_rate_to_pair"],
    ascending=False,
)
display(hard_pairs_df.head(20))
hard_pairs_df.to_csv(
    WORK_DIR / "hard_confusion_pairs.csv",
    index=False,
    encoding="utf-8-sig",
)

def pair_stats(cm_arr, pair):
    a, b = pair
    ia, ib = label_to_idx[a], label_to_idx[b]
    return {
        "pair": [a, b],
        f"{a}_total": int(cm_arr[ia].sum()),
        f"{a}_correct": int(cm_arr[ia, ia]),
        f"{a}_to_{b}": int(cm_arr[ia, ib]),
        f"{b}_total": int(cm_arr[ib].sum()),
        f"{b}_correct": int(cm_arr[ib, ib]),
        f"{b}_to_{a}": int(cm_arr[ib, ia]),
    }

focus_test_stats = pair_stats(cm, FOCUS_PAIR)

val_cm = confusion_matrix(
    val_y,
    val_pred,
    labels=np.arange(len(TARGET_GLOSSES)),
)
focus_val_stats = pair_stats(val_cm, FOCUS_PAIR)

focus_report = {
    "validation_clean": focus_val_stats,
    "test": focus_test_stats,
}
(WORK_DIR / "focus_pair_report.json").write_text(
    json.dumps(focus_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("FOCUS PAIR REPORT:")
print(json.dumps(focus_report, ensure_ascii=False, indent=2))


# Hard-cluster per-class diagnostics on clean validation and development test.
def per_class_f1_table(y, pred, split_name):
    vals = f1_score(
        y, pred, labels=np.arange(len(TARGET_GLOSSES)),
        average=None, zero_division=0
    )
    rows = []
    for g in HARD_CLUSTER:
        idx = label_to_idx[g]
        rows.append({
            "split": split_name,
            "gloss": g,
            "f1": float(vals[idx]),
        })
    return pd.DataFrame(rows)

hard_cluster_report_df = pd.concat([
    per_class_f1_table(val_y, val_pred, "validation_clean"),
    per_class_f1_table(y_true, y_pred, "development_test"),
], ignore_index=True)
display(hard_cluster_report_df)
hard_cluster_report_df.to_csv(
    WORK_DIR / "hard_cluster_report.csv", index=False, encoding="utf-8-sig"
)


In [ ]:
# ============================================================
# 22. V4.3 TRAINING CURVES
# ============================================================
hist = pd.DataFrame(history)

if len(hist):
    plt.figure(figsize=(9, 4))
    plt.plot(hist["epoch"], hist["train_loss"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Train loss")
    plt.title("V4.3 training loss")
    plt.tight_layout()
    plt.savefig(WORK_DIR / "training_loss.png", dpi=160)
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.plot(
        hist["epoch"],
        hist["val_macro_f1"],
        marker="o",
        label="Clean val Macro-F1",
    )
    if "stress_macro_f1" in hist:
        plt.plot(
            hist["epoch"],
            hist["stress_macro_f1"],
            marker="o",
            label="Stress val Macro-F1",
        )
    plt.plot(
        hist["epoch"],
        hist["selection_score"],
        marker="o",
        label="Selection score",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.ylim(0, 1.02)
    plt.title("V4.3 validation robustness")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        WORK_DIR / "validation_robustness.png",
        dpi=160,
    )
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.plot(
        hist["epoch"],
        hist["val_acc"],
        marker="o",
        label="Accuracy",
    )
    plt.plot(
        hist["epoch"],
        hist["val_top3"],
        marker="o",
        label="Top-3",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.ylim(0, 1.02)
    plt.title("Clean validation metrics")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        WORK_DIR / "validation_metrics.png",
        dpi=160,
    )
    plt.show()


if len(hist) and "val_hard_macro_f1" in hist:
    plt.figure(figsize=(9, 4))
    plt.plot(hist["epoch"], hist["val_hard_macro_f1"], marker="o", label="Hard mean F1")
    plt.plot(hist["epoch"], hist["val_hard_min_f1"], marker="o", label="Hard min F1")
    plt.xlabel("Epoch")
    plt.ylabel("F1")
    plt.ylim(0, 1.02)
    plt.title("V4.3 hard-cluster validation")
    plt.legend()
    plt.tight_layout()
    plt.savefig(WORK_DIR / "hard_cluster_validation.png", dpi=160)
    plt.show()


In [ ]:
# ============================================================
# 23. SAVE LABELS / CONFIG / FINAL SUMMARY
# ============================================================
label_map = {
    "label_to_idx": label_to_idx,
    "idx_to_label": {str(k): v for k, v in idx_to_label.items()},
}
(WORK_DIR / "label_map.json").write_text(
    json.dumps(label_map, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

summary = {
    "target_glosses": TARGET_GLOSSES,
    "num_classes": len(TARGET_GLOSSES),
    "canonical_train_before_val": len(train_all_df),
    "train_final": len(train_full_df),
    "validation": len(val_df),
    "test": len(test_df),
    "augmented_train_added": len(aug_df),
    "blacklist_unique_ids": sum(len(x) for x in BAD_VIDEO_IDS.values()),
    "mode": "keypoint_only_v4_3",
    "prediction_head": "main_only",
    "output_fusion": False,
    "stage3_enabled": False,
    "validation_split_method": VAL_SPLIT_METHOD,
    "signer_mapping_source": SIGNER_SOURCE,
    "signer_mapping_coverage": SIGNER_COVERAGE,
    "stress_validation": USE_STRESS_VAL,
    "training_scope": "30_target_glosses_only",
    "kp_reconstruction": USE_KP_RECONSTRUCTION,
    "active_trim": USE_ACTIVE_TRIM,
    "geometry_residual_bone_motion": USE_GEOMETRY_RESIDUAL,
    "focus_pair": list(FOCUS_PAIR),
    "focus_pair_ce_weight": FOCUS_PAIR_CE_WEIGHT,
    "focus_pair_margin_weight": FOCUS_PAIR_MARGIN_WEIGHT,
    "hard_cluster": HARD_CLUSTER,
    "hard_negative_map": HARD_NEGATIVE_MAP,
    "hard_negative_metric_weight": HARD_NEG_METRIC_WEIGHT,
    "hard_negative_margin": HARD_NEG_MARGIN,
    "hard_class_sampling_boosts": HARD_CLASS_SAMPLING_BOOSTS,
    "selection_weights": {
        "clean_macro": SELECT_CLEAN_MACRO_W,
        "stress_macro": SELECT_STRESS_MACRO_W,
        "hard_mean": SELECT_HARD_MEAN_W,
        "hard_min": SELECT_HARD_MIN_W,
    },
    "pretrain_checkpoint": str(PRETRAIN_PATH),
    "pretrained_tensors_loaded": len(loaded_report),
    "best_selection_score": best.get("best_selection_score"),
    "best_val_macro_f1": best.get("best_val_macro_f1"),
    "best_val_hard_macro_f1": best.get("best_val_hard_macro_f1"),
    "best_val_hard_min_f1": best.get("best_val_hard_min_f1"),
    "best_val_acc": best.get("best_val_acc"),
    "best_epoch": best.get("epoch"),
    "best_stage": best.get("stage"),
    "validation_metrics": val_metrics,
    "stress_validation_metrics": stress_val_metrics,
    "development_test_metrics": test_metrics,
    "test_metrics": test_metrics,
    "focus_pair_validation": focus_val_stats,
    "focus_pair_development_test": focus_test_stats,
    "methodology_note": (
        "V4.3 design was informed by earlier V4.2 test errors; treat this test as development test. "
        "Use a fresh holdout/unseen-signer set for an unbiased final accuracy claim."
    ),
    "references": {
        "v3_1": {
            "acc": CURRENT_V3_1_ACC,
            "macro_f1": CURRENT_V3_1_MACRO_F1,
            "top3": CURRENT_V3_1_TOP3,
        },
        "v4_1_main": {
            "acc": CURRENT_V4_1_MAIN_ACC,
            "macro_f1": CURRENT_V4_1_MAIN_MACRO_F1,
            "top3": CURRENT_V4_1_MAIN_TOP3,
        },
        "v4_2": {
            "acc": CURRENT_V4_2_ACC,
            "macro_f1": CURRENT_V4_2_MACRO_F1,
            "top3": CURRENT_V4_2_TOP3,
        },
    },
}

summary["delta_vs_v4_1_main"] = {
    "acc": float(test_metrics["acc"] - CURRENT_V4_1_MAIN_ACC),
    "macro_f1": float(test_metrics["macro_f1"] - CURRENT_V4_1_MAIN_MACRO_F1),
    "top3": float(test_metrics["top3"] - CURRENT_V4_1_MAIN_TOP3),
}
summary["delta_vs_v4_2"] = {
    "acc": float(test_metrics["acc"] - CURRENT_V4_2_ACC),
    "macro_f1": float(test_metrics["macro_f1"] - CURRENT_V4_2_MACRO_F1),
    "top3": float(test_metrics["top3"] - CURRENT_V4_2_TOP3),
}

summary["development_verdict"] = (
    "V4_3_BEATS_V4_1_MAIN_TOP1"
    if test_metrics["acc"] > CURRENT_V4_1_MAIN_ACC
    else (
        "V4_3_BETTER_MACRO_OR_TOP3_THAN_V4_1_MAIN"
        if (
            test_metrics["macro_f1"] > CURRENT_V4_1_MAIN_MACRO_F1
            or test_metrics["top3"] > CURRENT_V4_1_MAIN_TOP3
        )
        else "V4_3_NOT_BETTER_THAN_V4_1_MAIN"
    )
)

(WORK_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# ============================================================
# 24. V4.3 MODE CONFIRMATION
# ============================================================
print("KEYPOINT-ONLY V4.3")
print("- 30-class fine-tune only")
print("- NO 472-class domain adaptation")
print("- MAIN HEAD ONLY for final prediction")
print("- NO metric/hand output fusion")
print("- Stage 3 disabled")
print("- Small focus-pair CE/margin:", FOCUS_PAIR)
print("- Hard-negative metric cluster:", HARD_CLUSTER)
print("- Mild class sampling:", HARD_CLASS_SAMPLING_BOOSTS)
print("- Clean + deterministic stress + hard-class-aware checkpointing")
print("- Current test is development test for V4.3 iteration")


In [ ]:
# ============================================================
# 25. FINAL DEPLOYMENT EXPORT — BEST CHECKPOINT FROM THIS RUN
# ============================================================
# This cell does NOT use an old V4.3 checkpoint from /kaggle/input.
# It exports the best checkpoint produced above in WORK_DIR.

import copy
import hashlib
import zipfile
from pathlib import Path

import onnx
import onnxruntime as ort

EXPORT_THRESHOLD = 0.60
ONNX_OPSET = 17
ONNX_MAX_ABS_DIFF = 1e-3

# Reload the best checkpoint produced by THIS run.
best_export = torch.load(best_path, map_location="cpu", weights_only=False)
model.load_state_dict(best_export["model_state"], strict=True)
model.eval()

class VSL30MainExportWrapper(nn.Module):
    """Deployment graph = V4.3 keypoint encoder + MAIN classifier only."""
    def __init__(self, trained_model):
        super().__init__()
        # Deep-copy so exporting on CPU does not mutate the evaluation model.
        self.kp_encoder = copy.deepcopy(trained_model.kp_encoder).cpu().eval()
        self.classifier = copy.deepcopy(trained_model.classifier).cpu().eval()

    def forward(self, keypoints):
        # Important: return_parts=False -> auxiliary hand branch is not executed.
        z = self.kp_encoder(keypoints)
        return self.classifier(z)

deploy_model = VSL30MainExportWrapper(model).cpu().eval()

# Prefer a REAL preprocessed validation batch for equivalence checks.
_export_batch = next(iter(val_loader))
sample_input = _export_batch["kp"][: min(4, len(_export_batch["kp"]))].detach().cpu().float()
if sample_input.ndim != 4 or tuple(sample_input.shape[1:]) != (SEQ_LEN, KP_USED_POINTS, 4):
    raise RuntimeError(
        f"Unexpected export input shape {tuple(sample_input.shape)}; "
        f"expected [B,{SEQ_LEN},{KP_USED_POINTS},4]"
    )

with torch.inference_mode():
    ref_logits = deploy_model(sample_input)

# -----------------------------
# A. Save deployment-only PT
# -----------------------------
deploy_pt_path = WORK_DIR / "vsl30_v4_3_main_head_state.pt"
torch.save({
    "model_state": deploy_model.state_dict(),
    "source_best_checkpoint": best_path.name,
    "best_epoch": best_export.get("epoch"),
    "best_stage": best_export.get("stage"),
    "target_glosses": TARGET_GLOSSES,
    "input_shape": [None, SEQ_LEN, KP_USED_POINTS, 4],
    "prediction_head": "main_only",
    "confidence_threshold_default": EXPORT_THRESHOLD,
    "preprocessing": {
        "kp_reconstruction": USE_KP_RECONSTRUCTION,
        "seq_len": SEQ_LEN,
        "kp_used_points": KP_USED_POINTS,
        "coord_clip": COORD_CLIP,
        "active_trim": USE_ACTIVE_TRIM,
    },
}, deploy_pt_path)

# -----------------------------
# B. TorchScript
# -----------------------------
ts_path = WORK_DIR / "vsl30_v4_3_main_head.ts"
with torch.inference_mode():
    traced = torch.jit.trace(
        deploy_model,
        sample_input[:1],
        check_trace=True,
        strict=False,
    )
    traced = torch.jit.freeze(traced.eval())
    traced.save(str(ts_path))

loaded_ts = torch.jit.load(str(ts_path), map_location="cpu").eval()
with torch.inference_mode():
    ts_logits = loaded_ts(sample_input)

ts_max_diff = float((ref_logits - ts_logits).abs().max().item())
ts_same_top1 = bool(
    torch.equal(ref_logits.argmax(dim=1), ts_logits.argmax(dim=1))
)
print("TorchScript max_abs_diff:", ts_max_diff)
print("TorchScript Top-1 identical:", ts_same_top1)

if (not ts_same_top1) or ts_max_diff > 1e-4:
    raise RuntimeError("TorchScript equivalence check FAILED")

# -----------------------------
# C. ONNX
# -----------------------------
onnx_path = WORK_DIR / "vsl30_v4_3_main.onnx"

# Transformer/MHA tracing is more predictable with the native MHA fastpath disabled.
if hasattr(torch.backends, "mha") and hasattr(torch.backends.mha, "set_fastpath_enabled"):
    torch.backends.mha.set_fastpath_enabled(False)

with torch.inference_mode():
    torch.onnx.export(
        deploy_model,
        sample_input[:1],
        str(onnx_path),
        input_names=["keypoints"],
        output_names=["logits"],
        dynamic_axes={
            "keypoints": {0: "batch"},
            "logits": {0: "batch"},
        },
        opset_version=ONNX_OPSET,
        do_constant_folding=True,
        dynamo=False,
    )

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

ort_sess = ort.InferenceSession(
    str(onnx_path),
    providers=["CPUExecutionProvider"],
)
ort_logits = ort_sess.run(
    ["logits"],
    {"keypoints": sample_input.numpy()},
)[0]

ref_np = ref_logits.detach().cpu().numpy()
onnx_max_diff = float(np.max(np.abs(ref_np - ort_logits)))
onnx_same_top1 = bool(
    np.array_equal(ref_np.argmax(axis=1), ort_logits.argmax(axis=1))
)

print("ONNX max_abs_diff:", onnx_max_diff)
print("ONNX Top-1 identical:", onnx_same_top1)

if (not onnx_same_top1) or onnx_max_diff > ONNX_MAX_ABS_DIFF:
    raise RuntimeError(
        f"ONNX equivalence check FAILED: diff={onnx_max_diff:.6g}, "
        f"same_top1={onnx_same_top1}"
    )

# -----------------------------
# D. Deployment metadata
# -----------------------------
deployment_config = {
    "model_name": "VSL30_V4_3_FINAL",
    "source_checkpoint": best_path.name,
    "best_epoch": int(best_export.get("epoch", -1)),
    "best_stage": int(best_export.get("stage", -1)),
    "prediction_head": "main_only",
    "num_classes": len(TARGET_GLOSSES),
    "target_glosses": TARGET_GLOSSES,
    "input_name": "keypoints",
    "output_name": "logits",
    "input_shape": ["batch", SEQ_LEN, KP_USED_POINTS, 4],
    "input_dtype": "float32",
    "onnx_opset": ONNX_OPSET,
    "confidence_threshold_default": EXPORT_THRESHOLD,
    "threshold_note": "Default deployment reject threshold; do not tune on development test.",
    "torchscript_equivalence": {
        "max_abs_diff": ts_max_diff,
        "top1_identical": ts_same_top1,
    },
    "onnx_equivalence": {
        "max_abs_diff": onnx_max_diff,
        "top1_identical": onnx_same_top1,
    },
    "preprocessing": {
        "sequence_length": SEQ_LEN,
        "keypoints": KP_USED_POINTS,
        "channels": 4,
        "kp_reconstruction": USE_KP_RECONSTRUCTION,
        "active_trim": USE_ACTIVE_TRIM,
        "coord_clip": COORD_CLIP,
        "horizontal_flip": False,
    },
}
deployment_config_path = WORK_DIR / "deployment_config.json"
deployment_config_path.write_text(
    json.dumps(deployment_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Save a small export report.
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

export_report = {
    "best_checkpoint": {
        "path": best_path.name,
        "epoch": best_export.get("epoch"),
        "stage": best_export.get("stage"),
        "sha256": sha256_file(best_path),
    },
    "deploy_pt": {
        "path": deploy_pt_path.name,
        "size_mb": deploy_pt_path.stat().st_size / 1024**2,
        "sha256": sha256_file(deploy_pt_path),
    },
    "torchscript": {
        "path": ts_path.name,
        "size_mb": ts_path.stat().st_size / 1024**2,
        "sha256": sha256_file(ts_path),
        "max_abs_diff": ts_max_diff,
        "top1_identical": ts_same_top1,
    },
    "onnx": {
        "path": onnx_path.name,
        "size_mb": onnx_path.stat().st_size / 1024**2,
        "sha256": sha256_file(onnx_path),
        "max_abs_diff": onnx_max_diff,
        "top1_identical": onnx_same_top1,
        "opset": ONNX_OPSET,
    },
}
export_report_path = WORK_DIR / "export_report.json"
export_report_path.write_text(
    json.dumps(export_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# -----------------------------
# E. Bundle all final files
# -----------------------------
bundle_path = WORK_DIR / "vsl30_v4_3_FINAL_bundle.zip"

bundle_candidates = [
    best_path,
    deploy_pt_path,
    ts_path,
    onnx_path,
    WORK_DIR / "label_map.json",
    WORK_DIR / "deployment_config.json",
    WORK_DIR / "export_report.json",
    WORK_DIR / "summary.json",
    WORK_DIR / "classification_report.csv",
    WORK_DIR / "confusion_matrix.csv",
    WORK_DIR / "training_history.csv",
]

with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in bundle_candidates:
        if path.exists():
            zf.write(path, arcname=path.name)

print()
print("=" * 80)
print("FINAL EXPORT PASS")
print("=" * 80)
print("Best checkpoint :", best_path)
print("Deploy PT       :", deploy_pt_path)
print("TorchScript     :", ts_path)
print("ONNX            :", onnx_path)
print("Bundle          :", bundle_path)
print("Input shape     :", ["B", SEQ_LEN, KP_USED_POINTS, 4])
print("Output          :", ["B", len(TARGET_GLOSSES)], "logits")
print("Default threshold:", EXPORT_THRESHOLD)


In [ ]:
# ============================================================
# 26. OUTPUT FILES — V4.3 FINAL
# ============================================================
print("WORK_DIR:",WORK_DIR)
for p in sorted(WORK_DIR.iterdir()):
    if p.is_file(): print(f"{p.name:44s} {p.stat().st_size/1024**2:.2f} MB")
print("\nFINAL model files expected:")
for name in [
    "best_vsl30_v4_3.pt",
    "vsl30_v4_3_main_head_state.pt",
    "vsl30_v4_3_main_head.ts",
    "vsl30_v4_3_main.onnx",
    "deployment_config.json",
    "export_report.json",
    "vsl30_v4_3_FINAL_bundle.zip",
]:
    p = WORK_DIR / name
    print("  ", name, "OK" if p.exists() else "MISSING")
print("\nOn Kaggle: Save Version to preserve outputs.")


## Sau khi Run All xong

Output nằm tại:

```text
/kaggle/working/vsl30_v4_3/
```

### Model được tạo tự động từ chính lần train này

```text
best_vsl30_v4_3.pt
vsl30_v4_3_main_head_state.pt
vsl30_v4_3_main_head.ts
vsl30_v4_3_main.onnx
deployment_config.json
export_report.json
label_map.json
vsl30_v4_3_FINAL_bundle.zip
```

Notebook tự kiểm tra:

- checkpoint được load lại từ best epoch của lần chạy hiện tại;
- PyTorch ↔ TorchScript phải cùng Top-1;
- PyTorch ↔ ONNX Runtime phải cùng Top-1;
- ONNX `max_abs_diff <= 1e-3`;
- nếu equivalence check fail, cell export sẽ dừng thay vì đưa model sai.

### Input vẫn chỉ cần

1. Dataset `vsl-vietnamese-sign-language-v2`.
2. Pretrained encoder `best_vsl_metric_encoder.pt`.

**Không add `best_vsl30_v4_3.pt` cũ làm Input.** Notebook sẽ train V4.3 lại và tự tạo checkpoint/export mới.
